Imports

In [ ]:
import pandas as pd
import re
import sys
import numpy as np
from pathlib import Path
from sklearn.model_selection import GroupShuffleSplit
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error



Load Data

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "README.md").exists() and (PROJECT_ROOT.parent / "README.md").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "dubizzle_phones_clean.csv"
if RAW_DATA_PATH.exists():
    df = pd.read_csv(RAW_DATA_PATH)
else:
    try:
        from google.colab import files
    except ModuleNotFoundError as exc:
        raise FileNotFoundError(
            f"Place dubizzle_phones_clean.csv at {RAW_DATA_PATH} or run this notebook in Google Colab and upload it."
        ) from exc
    uploaded = files.upload()
    if not uploaded:
        raise FileNotFoundError("No CSV file was uploaded.")
    uploaded_name = next(iter(uploaded))
    df = pd.read_csv(uploaded_name)


Saving dubizzle_phones_clean.csv to dubizzle_phones_clean.csv


Dataset Shape

In [ ]:
print("Dataset shape:", df.shape)

Dataset shape: (8213, 30)


Dataset Columns

In [ ]:
print("Columns names:\n", df.columns.to_list())

Columns names:
 ['title', 'price', 'location', 'ad_url', 'ad_id', 'run_id', 'brand', 'storage_gb', 'ram_gb', 'storage_src', 'condition', 'condition_src', 'ram_src', 'has_5g', 'has_warranty', 'has_box', 'mentions_exchange', 'is_wanted_ad', 'is_accessory_hint', 'is_tablet_hint', 'is_parts_hint', 'accessory_first', 'bundle_hint', 'is_feature_phone_hint', 'price_placeholder', 'loc_area', 'loc_city', 'price_missing', 'price_suspicious', 'price_very_high']


In [ ]:
print("Number of columns:", len(df.columns))

Number of columns: 30


In [ ]:
df.head()

,title,price,location,ad_url,ad_id,run_id,brand,storage_gb,ram_gb,storage_src,...,is_parts_hint,accessory_first,bundle_hint,is_feature_phone_hint,price_placeholder,loc_area,loc_city,price_missing,price_suspicious,price_very_high
0,ايفون 13 iPhone,19500.0,مساكن المهندسين، مدينة نصر,/ad/%D8%A7%D9%8A%D9%81%D9%88%D9%86-13-iphone-I...,208854767,legacy,apple,NaN,NaN,NaN,...,False,False,False,False,False,مساكن المهندسين,مدينة نصر,False,False,False
1,Honor Magic V5,80000.0,اوبن اير مول، مدينتي,/ad/honor-magic-v5-ID209010621.html,209010621,legacy,honor,NaN,NaN,NaN,...,False,False,False,False,False,اوبن اير مول,مدينتي,False,False,False
2,OPPO Find X9 Pro ضمان محلي للبيع او البدل,57999.0,سموحة، الإسكندرية,/ad/oppo-find-x9-pro-%D8%B6%D9%85%D8%A7%D9%86-...,209003975,legacy,oppo,NaN,NaN,NaN,...,False,False,False,False,False,سموحة,الإسكندرية,False,False,False
3,Honor Robot Phone 512G 12G ram,94000.0,مدينة نصر، القاهرة,/ad/honor-robot-phone-512g-12g-ram-ID208841945...,208841945,legacy,honor,512.0,12.0,L1:unit,...,False,False,False,False,False,مدينة نصر,القاهرة,False,False,False
4,شاومي 17 الترا xiaomi 17 ultra,70000.0,التجمع الخامس، القاهرة الجديدة,/ad/%D8%B4%D8%A7%D9%88%D9%85%D9%8A-17-%D8%A7%D...,208281312,legacy,xiaomi,NaN,NaN,NaN,...,False,False,False,False,False,التجمع الخامس,القاهرة الجديدة,False,False,False


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8213 entries, 0 to 8212
Data columns (total 30 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   title                  8213 non-null   object 
 1   price                  8213 non-null   float64
 2   location               8207 non-null   object 
 3   ad_url                 8213 non-null   object 
 4   ad_id                  8213 non-null   int64  
 5   run_id                 8213 non-null   object 
 6   brand                  7772 non-null   object 
 7   storage_gb             586 non-null    float64
 8   ram_gb                 177 non-null    float64
 9   storage_src            586 non-null    object 
 10  condition              439 non-null    object 
 11  condition_src          439 non-null    object 
 12  ram_src                177 non-null    object 
 13  has_5g                 8213 non-null   bool   
 14  has_warranty           8213 non-null   bool   
 15  has_

In [ ]:
df.isna().sum().to_frame("Missing Values")

,Missing Values
title,0
price,0
location,6
ad_url,0
ad_id,0
run_id,0
brand,441
storage_gb,7627
ram_gb,8036
storage_src,7627


Data Types and Missing Values

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8213 entries, 0 to 8212
Data columns (total 30 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   title                  8213 non-null   object 
 1   price                  8213 non-null   float64
 2   location               8207 non-null   object 
 3   ad_url                 8213 non-null   object 
 4   ad_id                  8213 non-null   int64  
 5   run_id                 8213 non-null   object 
 6   brand                  7772 non-null   object 
 7   storage_gb             586 non-null    float64
 8   ram_gb                 177 non-null    float64
 9   storage_src            586 non-null    object 
 10  condition              439 non-null    object 
 11  condition_src          439 non-null    object 
 12  ram_src                177 non-null    object 
 13  has_5g                 8213 non-null   bool   
 14  has_warranty           8213 non-null   bool   
 15  has_

In [ ]:
df.isna().sum().to_frame("Number of Missing Values")

,Number of Missing Values
title,0
price,0
location,6
ad_url,0
ad_id,0
run_id,0
brand,441
storage_gb,7627
ram_gb,8036
storage_src,7627


Check Duplicate Titles

In [ ]:
df["title"].duplicated().sum()

np.int64(4513)

Inspect Duplicate Titles

In [ ]:
df.loc[df["title"].duplicated(keep = False) , "title"].head(20).to_frame()

,title
0,ايفون 13 iPhone
1,Honor Magic V5
5,Apple - iPhone 17 Pro
9,آبل - آيفون 15 برو ماكس
14,آبل - آيفون 17 برو ماكس
15,آبل - آيفون 13 برو ماكس
17,Apple - iPhone 15
18,Samsung S25 Ultra
19,سامسونج اس 24 الترا
21,Apple - iPhone 17 Pro Max


Inspect Title Examples

In [ ]:
df["title"].head(30)

,title
0,ايفون 13 iPhone
1,Honor Magic V5
2,OPPO Find X9 Pro ضمان محلي للبيع او البدل
3,Honor Robot Phone 512G 12G ram
4,شاومي 17 الترا xiaomi 17 ultra
5,Apple - iPhone 17 Pro
6,للبيع iPhone 15 Pro 1TB_ايفون 15 برو_زيرو بسعر...
7,Honor 500 pro
8,ايفون 11 بحاله الزيرو (iphone11) - سعر لقطه لل...
9,آبل - آيفون 15 برو ماكس


Check Title Normalization Examples

In [ ]:
df.loc[df["title"].str.contains("iPhone 17 Pro Max", case=False), "title"]

,title
21,Apple - iPhone 17 Pro Max
26,Apple - iPhone 17 Pro Max 512 GB
39,Apple - iPhone 17 Pro Max - Battery 98%
42,Apple - iPhone 17 Pro Max battery 100%
43,Apple iPhone 17 Pro Max (512GB) كسر زيرو | خال...
...,...
8049,Apple - iPhone 17 Pro Max
8057,Apple - iPhone 17 Pro Max
8072,Apple - iPhone 17 Pro Max battery 98% still in...
8084,Apple - iPhone 17 Pro Max


Check Title Variations

In [ ]:
df.loc[df["title"].str.contains("iPhone 17 Pro Max", case=False),"title"].nunique()

46

Inspect Unique Title Variations

In [ ]:
df.loc[df["title"].str.contains("iPhone 17 Pro Max", case=False),"title"].drop_duplicates().to_list()

['Apple - iPhone 17 Pro Max',
 'Apple - iPhone 17 Pro Max 512 GB',
 'Apple - iPhone 17 Pro Max - Battery 98%',
 'Apple - iPhone 17 Pro Max battery 100%',
 'Apple iPhone 17 Pro Max (512GB) كسر زيرو | خالص ضريبة',
 'iPhone 17 pro max',
 'Apple - iPhone 17 Pro Max First high copy',
 'Apple - iPhone 17 Pro Max -Used Like New',
 'آبل - آيفون 17 برو ماكس iphone 17 pro max',
 'Apple - iPhone 17 Pro max',
 'iPhone 17 Pro Max 512gb',
 'iPhone 17 Pro Max 256GB – Silver',
 'iPhone 17 pro max 256 سيلفر',
 'IPhone 17 Pro Max silver zero scratches',
 'آيفون 17 برو ماكس  سيلفر - Apple iPhone 17 Pro Max siliver',
 'آبل IPhone 17 Pro max',
 'Apple - iPhone 17 Pro Max خربوش بسيط جدا جدا ف الباغه',
 'Apple - iPhone 17 Pro Max tradeline جديد متبرشم معفي ضريبة',
 'IPhone 17 pro max',
 'Iphone 17 pro max blue new',
 'iPhone 17 Pro Max 256 شريحه نسخه محلي مسجل بالعلبه فقط',
 'iphone 17 pro max',
 'iPhone 17 Pro Max كسر زيرو بدون خربوش للبيع',
 'iPhone 17 Pro Max 256GB Deep Blue – 100% Battery',
 'Apple - iPh

Title Normalization

In [ ]:
df["title_normalized"] = df["title"].str.lower().str.strip()

In [ ]:
df[["title", "title_normalized"]].head(20)

,title,title_normalized
0,ايفون 13 iPhone,ايفون 13 iphone
1,Honor Magic V5,honor magic v5
2,OPPO Find X9 Pro ضمان محلي للبيع او البدل,oppo find x9 pro ضمان محلي للبيع او البدل
3,Honor Robot Phone 512G 12G ram,honor robot phone 512g 12g ram
4,شاومي 17 الترا xiaomi 17 ultra,شاومي 17 الترا xiaomi 17 ultra
5,Apple - iPhone 17 Pro,apple - iphone 17 pro
6,للبيع iPhone 15 Pro 1TB_ايفون 15 برو_زيرو بسعر...,للبيع iphone 15 pro 1tb_ايفون 15 برو_زيرو بسعر...
7,Honor 500 pro,honor 500 pro
8,ايفون 11 بحاله الزيرو (iphone11) - سعر لقطه لل...,ايفون 11 بحاله الزيرو (iphone11) - سعر لقطه لل...
9,آبل - آيفون 15 برو ماكس,آبل - آيفون 15 برو ماكس


Compare Unique Titles

In [ ]:
print("Original unique titles:", df["title"].nunique())
print("Normalized unique titles:", df["title_normalized"].nunique())

Original unique titles: 3700
Normalized unique titles: 3565


Normalize Arabic Letter Variants

In [ ]:
df["title_normalized"] = (df["title_normalized"]
    .str.replace("أ" , "ا", regex=False)
    .str.replace("إ" , "ا", regex=False)
    .str.replace("آ" , "ا", regex=False)
    .str.replace("ى" , "ي", regex=False)
)

Compare Normalized Titles

In [ ]:
df["title_normalized"].nunique()

3551

Inspect Title Punctuation

In [ ]:
df["title_normalized"].head(20).to_list()

['ايفون 13 iphone',
 'honor magic v5',
 'oppo find x9 pro ضمان محلي للبيع او البدل',
 'honor robot phone 512g 12g ram',
 'شاومي 17 الترا xiaomi 17 ultra',
 'apple - iphone 17 pro',
 'للبيع iphone 15 pro 1tb_ايفون 15 برو_زيرو بسعر لقطة لسرعه البيع ممتاز',
 'honor 500 pro',
 'ايفون 11 بحاله الزيرو (iphone11) - سعر لقطه للجادين للسرعه البيع 64 gb',
 'ابل - ايفون 15 برو ماكس',
 'huawei nova 15 max',
 'ايفون 16e جديد وارد الامارات مدفوع ضريبة | iphone 16e phone new',
 'oppo find x5 5g dual sim 256/8g black بحالة ممتازة',
 'samsung s24 ultra زيرو 256gb تيتانيوم مدفوع الضريبة بالضمان',
 'ابل - ايفون 17 برو ماكس',
 'ابل - ايفون 13 برو ماكس',
 'honor magic v5 dual sim 512/16g white gold جديد بضمان الوكيل',
 'apple - iphone 15',
 'samsung s25 ultra',
 'سامسونج اس 24 الترا']

Punctuation Normalization

In [ ]:
df["title_punct_normalized"] = (
    df["title_normalized"]
    .str.replace("-", " ", regex=False)
    .str.replace("_", " ", regex=False)
    .str.strip()
    .str.split()
    .str.join(" ")
)

Compare Titles After Punctuation Normalization

In [ ]:
print("Before:", df["title_normalized"].nunique())
print("After:", df["title_punct_normalized"].nunique())

Before: 3551
After: 3531


Review Normalized Titles

In [ ]:
df[["title", "title_normalized", "title_punct_normalized"]].head(10)

,title,title_normalized,title_punct_normalized
0,ايفون 13 iPhone,ايفون 13 iphone,ايفون 13 iphone
1,Honor Magic V5,honor magic v5,honor magic v5
2,OPPO Find X9 Pro ضمان محلي للبيع او البدل,oppo find x9 pro ضمان محلي للبيع او البدل,oppo find x9 pro ضمان محلي للبيع او البدل
3,Honor Robot Phone 512G 12G ram,honor robot phone 512g 12g ram,honor robot phone 512g 12g ram
4,شاومي 17 الترا xiaomi 17 ultra,شاومي 17 الترا xiaomi 17 ultra,شاومي 17 الترا xiaomi 17 ultra
5,Apple - iPhone 17 Pro,apple - iphone 17 pro,apple iphone 17 pro
6,للبيع iPhone 15 Pro 1TB_ايفون 15 برو_زيرو بسعر...,للبيع iphone 15 pro 1tb_ايفون 15 برو_زيرو بسعر...,للبيع iphone 15 pro 1tb ايفون 15 برو زيرو بسعر...
7,Honor 500 pro,honor 500 pro,honor 500 pro
8,ايفون 11 بحاله الزيرو (iphone11) - سعر لقطه لل...,ايفون 11 بحاله الزيرو (iphone11) - سعر لقطه لل...,ايفون 11 بحاله الزيرو (iphone11) سعر لقطه للجا...
9,آبل - آيفون 15 برو ماكس,ابل - ايفون 15 برو ماكس,ابل ايفون 15 برو ماكس


Improve Punctuation Normalization

In [ ]:
df["title_punct_normalized"] = (
    df["title_normalized"]
    .str.replace("-", " ", regex=False)
    .str.replace("_", " ", regex=False)
    .str.strip()
)

In [ ]:
df.loc[df["title_normalized"] != df["title_punct_normalized"],["title_normalized", "title_punct_normalized"]].head(10)

,title_normalized,title_punct_normalized
5,apple - iphone 17 pro,apple iphone 17 pro
6,للبيع iphone 15 pro 1tb_ايفون 15 برو_زيرو بسعر...,للبيع iphone 15 pro 1tb ايفون 15 برو زيرو بسعر...
8,ايفون 11 بحاله الزيرو (iphone11) - سعر لقطه لل...,ايفون 11 بحاله الزيرو (iphone11) سعر لقطه لل...
9,ابل - ايفون 15 برو ماكس,ابل ايفون 15 برو ماكس
14,ابل - ايفون 17 برو ماكس,ابل ايفون 17 برو ماكس
15,ابل - ايفون 13 برو ماكس,ابل ايفون 13 برو ماكس
17,apple - iphone 15,apple iphone 15
21,apple - iphone 17 pro max,apple iphone 17 pro max
22,apple - iphone 14,apple iphone 14
26,apple - iphone 17 pro max 512 gb,apple iphone 17 pro max 512 gb


In [ ]:
print("Before:", df["title_normalized"].nunique())
print("After:", df["title_punct_normalized"].nunique())

Before: 3551
After: 3551


Normalize Extra Spaces

In [ ]:
df["title_punct_normalized"] = (
    df["title_punct_normalized"]
    .str.split()
    .str.join(" ")
)

In [ ]:
df["title_punct_normalized"].head(20).to_list()

['ايفون 13 iphone',
 'honor magic v5',
 'oppo find x9 pro ضمان محلي للبيع او البدل',
 'honor robot phone 512g 12g ram',
 'شاومي 17 الترا xiaomi 17 ultra',
 'apple iphone 17 pro',
 'للبيع iphone 15 pro 1tb ايفون 15 برو زيرو بسعر لقطة لسرعه البيع ممتاز',
 'honor 500 pro',
 'ايفون 11 بحاله الزيرو (iphone11) سعر لقطه للجادين للسرعه البيع 64 gb',
 'ابل ايفون 15 برو ماكس',
 'huawei nova 15 max',
 'ايفون 16e جديد وارد الامارات مدفوع ضريبة | iphone 16e phone new',
 'oppo find x5 5g dual sim 256/8g black بحالة ممتازة',
 'samsung s24 ultra زيرو 256gb تيتانيوم مدفوع الضريبة بالضمان',
 'ابل ايفون 17 برو ماكس',
 'ابل ايفون 13 برو ماكس',
 'honor magic v5 dual sim 512/16g white gold جديد بضمان الوكيل',
 'apple iphone 15',
 'samsung s25 ultra',
 'سامسونج اس 24 الترا']

Inspect Remaining Punctuation

In [ ]:
df.loc[df["title_punct_normalized"].str.contains(r"[|()]", regex=True, na=False), ["title", "title_punct_normalized"]].head(10)

,title,title_punct_normalized
8,ايفون 11 بحاله الزيرو (iphone11) - سعر لقطه لل...,ايفون 11 بحاله الزيرو (iphone11) سعر لقطه للجا...
11,ايفون 16e جديد وارد الامارات مدفوع ضريبة | iPh...,ايفون 16e جديد وارد الامارات مدفوع ضريبة | iph...
40,موبايل Samsung S23 Ultra مستعمل (256GB / 12GB ...,موبايل samsung s23 ultra مستعمل (256gb / 12gb ...
43,Apple iPhone 17 Pro Max (512GB) كسر زيرو | خال...,apple iphone 17 pro max (512gb) كسر زيرو | خال...
72,Infinix smart 20 (4+4/64) جديد,infinix smart 20 (4+4/64) جديد
196,ايفون ١٦ بلس (iphone 16 plus),ايفون ١٦ بلس (iphone 16 plus)
327,Apple iPhone 16 Pro (128 g) كسر زيرو | بطارية ...,apple iphone 16 pro (128 g) كسر زيرو | بطارية ...
527,Xiaomi Poco X3 (128/6),xiaomi poco x3 (128/6)
529,جهازين 11pro max (X),جهازين 11pro max (x)
653,iPhone 12 Pro Max (256) zero tax,iphone 12 pro max (256) zero tax


Normalize Arabic and English Digits

In [ ]:
digit_map = str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789")

df["title_normalized"] = (df["title_normalized"].str.translate(digit_map))

Remove Arabic Diacritics

In [ ]:
df["title_normalized"] = (df["title_normalized"].str.replace(r"[\u064B-\u065F\u0670]", "" , regex=True))

In [ ]:
df.loc[df["title"].str.contains("[ًٌٍَُِّْ]", na=False),["title", "title_normalized"]].head(10)

,title,title_normalized
354,سامسونج جالاكسي A07 مساحة 128GB رام 6GB - زير...,سامسونج جالاكسي a07 مساحة 128gb رام 6gb - زير...
507,Realme 9i 6/128 بحالة ممتازة جدًا + الكرتونة,realme 9i 6/128 بحالة ممتازة جدا + الكرتونة
746,للبيغ Infinix Smart بحالة ممتازة وبسعر لُقطة!,للبيغ infinix smart بحالة ممتازة وبسعر لقطة!
1499,لقطه العمر هاتف سامسونج جالاكسي ايه 13 ب 3000ج...,لقطه العمر هاتف سامسونج جالاكسي ايه 13 ب 3000ج...
2296,13 عاديً,13 عادي
2443,اوبو رينو 6 5g الفون زي الجديد حرفياً موضح بالصور,اوبو رينو 6 5g الفون زي الجديد حرفيا موضح بالصور
2495,​ فرصة مميزة للبيع: هاتف Infinix Note 50s كسر ...,​ فرصة مميزة للبيع: هاتف infinix note 50s كسر ...
3507,آبل - آيفون 17 برو ماكس زيرو تماماً جديد,ابل - ايفون 17 برو ماكس زيرو تماما جديد
3790,موبايل مختلف وفخم جدًا لعشاق التميز OPPO Find ...,موبايل مختلف وفخم جدا لعشاق التميز oppo find n...
4020,موبل سامسونج ِ30 جيد جدا للبيع,موبل سامسونج 30 جيد جدا للبيع


Compare Unique Titles Before and After Punctuation Normalization

In [ ]:
print("Unique normalized titles:", df["title_normalized"].nunique())
print("Unique punctuation-normalized titles:", df["title_punct_normalized"].nunique())

Unique normalized titles: 3521
Unique punctuation-normalized titles: 3531


Rebuild Punctuation-Normalized Titles

In [ ]:
df["title_punct_normalized"] = (df["title_normalized"]
    .str.replace(r"[-_]", " ", regex=True)
    .str.split()
    .str.join(" ")
)

In [ ]:
df[["title", "brand", "storage_gb", "ram_gb"]].head(20)

,title,brand,storage_gb,ram_gb
0,ايفون 13 iPhone,apple,NaN,NaN
1,Honor Magic V5,honor,NaN,NaN
2,OPPO Find X9 Pro ضمان محلي للبيع او البدل,oppo,NaN,NaN
3,Honor Robot Phone 512G 12G ram,honor,512.0,12.0
4,شاومي 17 الترا xiaomi 17 ultra,xiaomi,NaN,NaN
5,Apple - iPhone 17 Pro,apple,NaN,NaN
6,للبيع iPhone 15 Pro 1TB_ايفون 15 برو_زيرو بسعر...,apple,1024.0,NaN
7,Honor 500 pro,honor,NaN,NaN
8,ايفون 11 بحاله الزيرو (iphone11) - سعر لقطه لل...,apple,64.0,NaN
9,آبل - آيفون 15 برو ماكس,apple,NaN,NaN


In [ ]:
print("Unique normalized titles:", df["title_normalized"].nunique())
print("Unique punctuation-normalized titles:", df["title_punct_normalized"].nunique())

Unique normalized titles: 3521
Unique punctuation-normalized titles: 3500


In [ ]:
df[["title", "title_normalized", "title_punct_normalized"]].sample(10, random_state=42)

,title,title_normalized,title_punct_normalized
6791,فيفو x200 fe,فيفو x200 fe,فيفو x200 fe
7863,اوبو رينو 10 برو,اوبو رينو 10 برو,اوبو رينو 10 برو
3647,شاومي موديل اخر,شاومي موديل اخر,شاومي موديل اخر
7364,Apple - iPhone 17 Pro Max,apple - iphone 17 pro max,apple iphone 17 pro max
5549,آبل - آيفون 14 برو,ابل - ايفون 14 برو,ابل ايفون 14 برو
5584,آبل - آيفون 13,ابل - ايفون 13,ابل ايفون 13
2540,Apple - iPhone 15 Pro Max,apple - iphone 15 pro max,apple iphone 15 pro max
1882,Vivo y19 s Pro,vivo y19 s pro,vivo y19 s pro
970,آبل - آيفون 11 برو ماكس,ابل - ايفون 11 برو ماكس,ابل ايفون 11 برو ماكس
5952,Apple - iPhone 12 Pro Max,apple - iphone 12 pro max,apple iphone 12 pro max


Check for Extra Spaces in Normalized Titles

In [ ]:
df["title_punct_normalized"].str.contains(r"\s{2,}", regex=True, na=False).sum()

np.int64(0)

 Verify Final Title Normalization

In [ ]:
df[["title", "title_normalized", "title_punct_normalized"]].sample(10, random_state=42)

,title,title_normalized,title_punct_normalized
6791,فيفو x200 fe,فيفو x200 fe,فيفو x200 fe
7863,اوبو رينو 10 برو,اوبو رينو 10 برو,اوبو رينو 10 برو
3647,شاومي موديل اخر,شاومي موديل اخر,شاومي موديل اخر
7364,Apple - iPhone 17 Pro Max,apple - iphone 17 pro max,apple iphone 17 pro max
5549,آبل - آيفون 14 برو,ابل - ايفون 14 برو,ابل ايفون 14 برو
5584,آبل - آيفون 13,ابل - ايفون 13,ابل ايفون 13
2540,Apple - iPhone 15 Pro Max,apple - iphone 15 pro max,apple iphone 15 pro max
1882,Vivo y19 s Pro,vivo y19 s pro,vivo y19 s pro
970,آبل - آيفون 11 برو ماكس,ابل - ايفون 11 برو ماكس,ابل ايفون 11 برو ماكس
5952,Apple - iPhone 12 Pro Max,apple - iphone 12 pro max,apple iphone 12 pro max


Count Unique Titles After Final Normalization

In [ ]:
df["title_punct_normalized"].nunique()

3500

Create Final Normalized Title Groups

In [ ]:
df["title_group"] = df["title_punct_normalized"]

In [ ]:
print("Missing groups:", df["title_group"].isna().sum())
print("Total groups:", df["title_group"].nunique())
print("Total rows:", len(df))

Missing groups: 0
Total groups: 3500
Total rows: 8213


Validate Title Normalization

In [ ]:
print("Original unique titles:", df["title"].nunique())
print("Normalized unique titles:", df["title_normalized"].nunique())
print("Final title groups:", df["title_group"].nunique())
print("Missing groups:", df["title_group"].isna().sum())
print("Total rows:", len(df))

Original unique titles: 3700
Normalized unique titles: 3521
Final title groups: 3500
Missing groups: 0
Total rows: 8213


In [ ]:
import zipfile
zip_path = "/content/project_splits.zip"
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("/content/project_splits")

In [ ]:
train_df = pd.read_csv("/content/project_splits/data/processed/train_set.csv")
val_df = pd.read_csv("/content/project_splits/data/splits/validation.csv")
test_df = pd.read_csv("/content/project_splits/data/processed/frozen_test_set.csv")
stage1_df = train_df.copy()
stage1_val_df = val_df.copy()
stage1_test_df = test_df.copy()

In [ ]:
print(train_df.shape)
print(val_df.shape)
print(test_df.shape)

(5165, 33)
(1252, 33)
(1796, 33)


Create Stage 1 Training Dataset

Define Final iPhone Model Extraction Function


In [ ]:
def extract_iphone_model(title):
    text = re.sub(r"\s+", " ", str(title).lower()).strip()
    special_cases = [
        (r"(?:iphone|i\s*phone|ايفون|آيفون)\s*6\s*(?:s|اس|أس)\s*(?:plus|بلس)\b", "iphone 6s plus"),
        (r"(?:iphone|i\s*phone|ايفون|آيفون)\s*x\s*(?:s|اس|أس)\s*(?:max|ماكس)\b", "iphone xs max"),
        (r"(?:iphone|i\s*phone|ايفون|آيفون)\s*x\s*(?:max|ماكس)\b", "iphone x max"),
        (r"ip+h+o+n+e+\s*17\s+pro+m+a+x+", "iphone 17 pro max"),
        (r"ايفون+ن?\s*17\s*برماكس", "iphone 17 pro max"),
        (r"iphoneee?\s*17\s*pro+\s*ma+x+", "iphone 17 pro max"),
        (r"iphone+\s*17\s*pro+\s*ma+x+", "iphone 17 pro max"),
        (r"ايفون\s*17\s*برو+ما+كس", "iphone 17 pro max"),
        (r"ايفون\s*17\s*برماكس", "iphone 17 pro max"),
        (r"iphone\s*pro\s*max\s*17", "iphone 17 pro max"),
        (r"ايفون\s*pro\s*max\s*17", "iphone 17 pro max"),
        (r"iphone\s*3g\b", "iphone 3g"),
        (r"ايفون\s*3g\b", "iphone 3g"),
        (r"ايفون\s*6اس\b", "iphone 6s"),
        (r"ايفون\s*8بلص\b", "iphone 8 plus"),
        (r"ايفون\s*i\s*phone\s*plus\s*7", "iphone 7 plus"),
    ]

    for pattern, model in special_cases:
        if re.search(pattern, text):
            return model
    special_models = [
        (r"\bxs\s*max\b|\bاكس\s*اس\s*ماكس\b", "iphone xs max"),
        (r"\bxs\b|\bاكس\s*اس\b", "iphone xs"),
        (r"\bxr\b|\bاكس\s*ار\b", "iphone xr"),
        (r"\bx\b|\bاكس\b", "iphone x"),
        (r"\b(?:iphone|i\s*phone)\s*se\b", "iphone se"),
        (r"\bايفون\s*اس\s*اي\b", "iphone se"),
        (r"\bايفون\s*اكس\s*اس\s*ماكس\b", "iphone xs max"),
        (r"\bايفون\s*اكس\s*ار\b", "iphone xr"),
        (r"\bايفون\s*اكس\s*اس\b", "iphone xs"),
        (r"\bايفون\s*اكس\b", "iphone x"),
        (r"\b(?:iphone|i\s*phone)\s*xs\s*max\b", "iphone xs max"),
        (r"\b(?:iphone|i\s*phone)\s*xr\b", "iphone xr"),
        (r"\b(?:iphone|i\s*phone)\s*xs\b", "iphone xs"),
        (r"\b(?:iphone|i\s*phone)\s*x\b", "iphone x"),
    ]

    for pattern, model in special_models:
        if re.search(pattern, text):
            return model

    model_patterns = [
        (r"(?:iphone|i\s*phone|ايفون|آيفون)\s*(\d{1,2})\s*pro\s*max\b", "pro max"),
        (r"(?:iphone|i\s*phone|ايفون|آيفون)\s*(\d{1,2})\s*برو\s*ماكس\b", "pro max"),
        (r"(?:iphone|i\s*phone|ايفون|آيفون)\s*(\d{1,2})\s*plus\b", "plus"),
        (r"(?:iphone|i\s*phone|ايفون|آيفون)\s*(\d{1,2})\s*بلس\b", "plus"),
        (r"(?:iphone|i\s*phone|ايفون|آيفون)\s*(\d{1,2})\s*pro\b", "pro"),
        (r"(?:iphone|i\s*phone|ايفون|آيفون)\s*(\d{1,2})\s*برو\b", "pro"),
    ]

    for pattern, version in model_patterns:
        match = re.search(pattern, text)
        if match:
            return f"iphone {match.group(1)} {version}"

    numeric_pattern = re.compile(
        r"(?:\biphone\s*|\bi\s*phone\s*|ايفون\s*)"
        r"(\d{1,2})(s)?\s*"
        r"(pro\s*max|promax|pro|max|plus|mini|air|e)?\b"
    )

    match = numeric_pattern.search(text)

    if match:
        number = match.group(1)
        suffix = match.group(2) or ""
        version = re.sub(r"\s+", " ", (match.group(3) or "").strip())

        if version == "promax":
            version = "pro max"

        model = f"iphone {number}{suffix}"
        return f"{model} {version}" if version else model

    match = re.search(
        r"ايفون\s+(pro|max|plus|برو|ماكس|بلس)\s*(\d{1,2})",
        text
    )

    if match:
        version = {
            "برو": "pro",
            "ماكس": "max",
            "بلس": "plus"
        }.get(match.group(1), match.group(1))

        return f"iphone {match.group(2)} {version}"

    pattern = re.compile(
        r"\b(?:iphone\s*)?(\d{1,2})\s*"
        r"(pro\s*max|pro|max|plus|mini|air|e)?\b"
        r"|ايفون\s*(\d{1,2})\s*"
        r"(برو\s*ماكس|برو|ماكس|بلس|ميني|اير|e)?"
    )

    match = pattern.search(text)

    if not match:
        return "unknown"

    number = match.group(1) or match.group(3)
    version = (match.group(2) or match.group(4) or "").strip()

    version = {
        "برو ماكس": "pro max",
        "برو": "pro",
        "ماكس": "max",
        "بلس": "plus",
        "ميني": "mini",
        "اير": "air"
    }.get(version, version)

    return f"iphone {number} {version}".strip()

Extract iPhone Models and Evaluate Coverage


In [ ]:
apple_titles = stage1_df[stage1_df["brand"] == "apple"].copy()

apple_titles["iphone_model"] = apple_titles[
    "title_punct_normalized"
].apply(extract_iphone_model)

print("Total Apple ads:", len(apple_titles))
print("Extracted:", (apple_titles["iphone_model"] != "unknown").sum())
print("Unknown:", (apple_titles["iphone_model"] == "unknown").sum())
print(
    "Coverage:",
    round((apple_titles["iphone_model"] != "unknown").mean() * 100, 2),
    "%"
)

display(
    apple_titles[
        ["title", "title_punct_normalized", "iphone_model"]
    ].head(20)
)

Total Apple ads: 2462
Extracted: 2443
Unknown: 19
Coverage: 99.23 %


,title,title_punct_normalized,iphone_model
0,ايفون 13 iPhone,ايفون 13 iphone,iphone 13
5,للبيع iPhone 15 Pro 1TB_ايفون 15 برو_زيرو بسعر...,للبيع iphone 15 pro 1tb ايفون 15 برو زيرو بسعر...,iphone 15 pro
6,ايفون 11 بحاله الزيرو (iphone11) - سعر لقطه لل...,ايفون 11 بحاله الزيرو (iphone11) سعر لقطه للجا...,iphone 11
7,آبل - آيفون 15 برو ماكس,ابل ايفون 15 برو ماكس,iphone 15 pro max
9,ايفون 16e جديد وارد الامارات مدفوع ضريبة | iPh...,ايفون 16e جديد وارد الامارات مدفوع ضريبة | iph...,iphone 16 e
12,آبل - آيفون 17 برو ماكس,ابل ايفون 17 برو ماكس,iphone 17 pro max
16,Apple - iPhone 17 Pro Max,apple iphone 17 pro max,iphone 17 pro max
17,Apple - iPhone 14,apple iphone 14,iphone 14
20,Apple - iPhone 17 Pro Max 512 GB,apple iphone 17 pro max 512 gb,iphone 17 pro max
23,iPhone 16 pro max,iphone 16 pro max,iphone 16 pro max


Save Final iPhone Predictions

In [ ]:
iphone_model_compact = apple_titles["iphone_model"].copy()
print("Saved predictions:", len(iphone_model_compact))
print("Extracted:", (iphone_model_compact != "unknown").sum())
print("Unknown:", (iphone_model_compact == "unknown").sum())

Saved predictions: 2462
Extracted: 2443
Unknown: 19


Add iPhone Models to Stage 1 Dataset

In [ ]:
apple_mask = stage1_df["brand"] == "apple"
stage1_df["iphone_model"] = "not_apple"

stage1_df.loc[apple_mask, "iphone_model"] = (
    stage1_df.loc[apple_mask, "title_punct_normalized"]
    .apply(extract_iphone_model)
)

print("iPhone column added successfully.")
print("Apple ads:", apple_mask.sum())
print(
    "Unknown iPhone models:",
    (stage1_df.loc[apple_mask, "iphone_model"] == "unknown").sum()
)

display(
    stage1_df.loc[
        apple_mask,
        ["title", "iphone_model"]
    ].head(10)
)

iPhone column added successfully.
Apple ads: 2462
Unknown iPhone models: 19


,title,iphone_model
0,ايفون 13 iPhone,iphone 13
5,للبيع iPhone 15 Pro 1TB_ايفون 15 برو_زيرو بسعر...,iphone 15 pro
6,ايفون 11 بحاله الزيرو (iphone11) - سعر لقطه لل...,iphone 11
7,آبل - آيفون 15 برو ماكس,iphone 15 pro max
9,ايفون 16e جديد وارد الامارات مدفوع ضريبة | iPh...,iphone 16 e
12,آبل - آيفون 17 برو ماكس,iphone 17 pro max
16,Apple - iPhone 17 Pro Max,iphone 17 pro max
17,Apple - iPhone 14,iphone 14
20,Apple - iPhone 17 Pro Max 512 GB,iphone 17 pro max
23,iPhone 16 pro max,iphone 16 pro max


Shared model extraction helpers

In [ ]:
ARABIC_DIGITS = str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789")
BRAND_WORDS = {
    "شاومي": "xiaomi", "ريدمي": "redmi", "ردمي": "redmi", "بوكو": "poco",
    "اوبو": "oppo", "رينو": "reno", "فايند": "find",
    "ريلمي": "realme", "ريالمي": "realme", "نارزو": "narzo",
    "هونر": "honor", "هونور": "honor", "اونر": "honor", "ماجيك": "magic",
    "انفينكس": "infinix", "انفنكس": "infinix",
    "فيفو": "vivo", "هواوي": "huawei", "نوفا": "nova",
    "موتورولا": "motorola", "موتو": "moto", "ريزر": "razr",
    "ون بلس": "oneplus", "وان بلس": "oneplus", "ونبلس": "oneplus", "نورد": "nord",
    "تكنو": "tecno", "سبارك": "spark", "كامون": "camon", "فانتوم": "phantom", "بوفا": "pova",
    "ايتل": "itel", "جوجل": "google", "بيكسل": "pixel", "سوني": "sony",
    "اكسبيريا": "xperia", "اكسبريا": "xperia", "نوكيا": "nokia", "لينوفو": "lenovo",
    "انفنيكس": "infinix", "نوبيا": "nubia", "بورا": "pura", "ستايلس": "stylus",
    "رازر": "razr", "بكسل": "pixel",
}

SERIES_WORDS = {
    "برو": "pro", "بلس": "plus", "الترا": "ultra", "ماكس": "max", "لايت": "lite",
    "نوت": "note", "ميني": "mini", "فولد": "fold", "فليب": "flip",
    "بلاي": "play", "برايم": "prime",
    "هوت": "hot", "سمارت": "smart", "ايدج": "edge",
}

ARABIC_RANGE = "\u0600-\u06FF"


def prepare_text(title, letters=None, aliases=None, pre=None):
    text = str(title).lower().translate(ARABIC_DIGITS)
    text = re.sub(r"[\u064B-\u065F\u0670\u0640]", "", text)
    for ch in "أإآ":
        text = text.replace(ch, "ا")
    text = text.replace("ى", "ي")
    text = text.replace("+", " plus ").replace("_", " ")
    text = re.sub(r"[^\w\s\.]", " ", text)
    text = re.sub(r"(?<!\d)\.|\.(?!\d)", " ", text)
    text = re.sub(
        rf"(?<=[{ARABIC_RANGE}])(?=[a-z0-9])|(?<=[a-z0-9])(?=[{ARABIC_RANGE}])",
        " ", text
    )

    all_brand_words = {**BRAND_WORDS, **(aliases or {})}
    for old in sorted(all_brand_words, key=len, reverse=True):
        text = text.replace(old, f" {all_brand_words[old]} ")

    for old, new in SERIES_WORDS.items():
        text = re.sub(rf"(?<![{ARABIC_RANGE}]){old}(?![{ARABIC_RANGE}])", f" {new} ", text)

    text = re.sub(r"\s+", " ", text).strip()

    text = re.sub(r"\bnot\s*(?=\d)", "note ", text)                      # not 10 -> note 10
    text = re.sub(r"(?<=\d)(pro|max|plus|ultra|lite|mini)\b", r" \1", text)  # 14pro -> 14 pro
    text = re.sub(r"(?<=\d)bro\b", " pro", text)                           # x3bro -> x3 pro
    text = re.sub(r"(?<=[a-z])5g\b", " 5g", text)                          # 50s5g -> 50s 5g

    for pattern, repl in (pre or []):
        text = re.sub(pattern, repl, text)

    for old, new in (letters or {}).items():
        text = re.sub(rf"(?<!\S){old}\s*(?=\d)", f" {new} ", text)
        text = re.sub(rf"(?<=\d)\s*{old}(?![{ARABIC_RANGE}])", f"{new} ", text)

    return re.sub(r"\s+", " ", text).strip()


BRAND_CONFIG = {}


def extract_brand_model(title, brand):
    cfg = BRAND_CONFIG[brand]
    raw = str(title).lower()
    if len(brand) >= 4:
        raw = re.sub(rf"({brand})(?=[a-z0-9])", r"\1 ", raw)
    text = prepare_text(raw, cfg.get("letters"), cfg.get("aliases"), cfg.get("pre"))

    for pattern, template in cfg["rules"]:
        match = re.search(pattern, text)
        if match:
           groups = [g or "" for g in match.groups()]
           model = template.format("", *groups)

           model = re.sub(r"\s+", " ", model).strip()
           model = re.sub(r"\b(pro|max|plus|ultra|lite|mini)\b", r"\1", model)

           return f"{brand} {model}".strip()

    return "unknown"





Explore Samsung Listings

In [ ]:
samsung_titles = stage1_df[
    stage1_df["brand"] == "samsung"
].copy()

print("Total Samsung ads:", len(samsung_titles))

display(
    samsung_titles[
        ["title", "title_punct_normalized"]
    ].sample(
        n=min(20, len(samsung_titles)),
        random_state=10
    )
)

Total Samsung ads: 749


,title,title_punct_normalized
1953,Samsung A26,samsung a26
4913,Samsung A7,samsung a7
1477,Samsung A17. سامسونج رامات ٨ -٢٥٦,samsung a17. سامسونج رامات 8 256
3362,سامسونج ايه 51,سامسونج ايه 51
3427,سامسونج ايه 51 حاله نادره بدون صيانه بالعلبه و...,سامسونج ايه 51 حاله نادره بدون صيانه بالعلبه و...
3428,موبايل سامسونج J6 Plus بعلبته وحالته شبه جديد ...,موبايل سامسونج j6 plus بعلبته وحالته شبه جديد ...
1944,سامسونج زد فولد 8,سامسونج زد فولد 8
4774,سامسونج ايه 23,سامسونج ايه 23
4976,Samsung Z Fold 4 dual sim 256gb needs a int...,samsung z fold 4 dual sim 256gb needs a intern...
1702,سامسونج موديل اخر A17,سامسونج موديل اخر a17


Define Final Samsung Model Extraction Rules

In [ ]:
BRAND_CONFIG["samsung"] = {
    "letters": {
        "ايه": "a",
        "اس": "s",
        "ام": "m",
        "اف": "f",
        "جيه": "j", "جي": "j", "چي": "j",
        "زد": "z",
    },

    "aliases": {
        "جالكسي": "galaxy", "جلاكسي": "galaxy", "جالاكسي": "galaxy",
        "سامسونج": "samsung",
        "جراند": "grand",
    },

    "pre": [
        (r"galaxy(?=[a-z0-9])", "galaxy "),
        (r"(?<![\u0600-\u06FF])اف\s*اي(?![\u0600-\u06FF])", " fe "),
        (r"a⁰", " "),
        (r"[⁰⁵⁷]", lambda m: {"⁰": "0", "⁵": "5", "⁷": "7"}[m.group()]),
        (r"(?<![\u0600-\u06FF])اي\s*(?=\d)", " a "),
        (r"(?<!\d)5\s*(?:g|جي)\b", " "),
    ],

    "rules": [
        (r"tri\s*fold|ترايفولد", "z trifold"),
        (r"\bz\s*fold\s*(\d+)\b", "z fold{1}"),
        (r"\bz\s*flip\s*(\d+)\b", "z flip{1}"),
        (r"\bfold\s*(\d+)\b", "z fold{1}"),
        (r"\bflip\s*(\d+)\b", "z flip{1}"),
        (r"\bx\s*cover\s*(\d+)\s*(pro)?\b", "xcover{1} {2}"),
        (r"\b26\s*ultra\b", "s26 ultra"),
        (r"\ba\s*03\s*core\b", "a03 core"),
        (r"\ba\s*04e\b", "a04e"),
        (r"\bgrand\s*prime\s*pro\b", "grand prime pro"),
        (r"\bgrand\s*prime\b", "grand prime"),
        (r"\bmega\b", "mega"),
        (r"\bnote\s*(\d+)\s*(ultra|plus)?\b", "note{1} {2}"),
        (r"\bs\s*(\d+)\s*(ultra|plus|fe)?\b", "s{1} {2}"),
        (r"\ba\s*(\d{1,3})\s*s\b", "a{1}s"),
        (r"\ba\s*(\d{1,3})\s*(ultra|plus)?\b", "a{1} {2}"),
        (r"\bf\s*(\d{1,3})\b", "f{1}"),
        (r"\bm\s*(\d{1,3})\s*s\b", "m{1}s"),
        (r"\bm\s*(\d{1,3})\s*(ultra|plus)?\b", "m{1} {2}"),
        (r"\bj\s*(\d{1,2})\s*(prime|plus|pro)\b", "j{1} {2}"),   # لاحقة في أي مكان (J7 ... J7 prime)
        (r"\bj\s*(\d{1,2})\b", "j{1}"),
        (r"\b(\d{1,2})\s*fe\b", "s{1} fe"),
        (r"\b(\d{1,3})\s*a\b", "a{1}"),
        (r"\bfold\b", "z fold"),
        (r"\bgrand\b", "grand"),
        (r"^(?:(?:samsung|galaxy|موبايل|موبيل|موبل|تليفون|للبيع|بيع|جديد|مستعمل|other|model|موديل|اخر)\s*)*$", "other"),
    ],
}

Test Special Samsung Model Cases

In [ ]:
samsung_test_titles = [
    "Samsung Galaxy S23 Ultra",
    "سامسونج جالكسي A54",
    "Galaxy Z Fold 5",
    "Samsung Galaxy A03 Core",
    "Samsung Note 20 Ultra",
]

for title in samsung_test_titles:
    print(title, "->", extract_brand_model(title, "samsung"))

Samsung Galaxy S23 Ultra -> samsung s23 ultra
سامسونج جالكسي A54 -> samsung a54
Galaxy Z Fold 5 -> samsung z fold5
Samsung Galaxy A03 Core -> samsung a03 core
Samsung Note 20 Ultra -> samsung note20 ultra


 Extract Samsung Models and Evaluate Coverage

In [ ]:
samsung_titles = stage1_df.loc[
    stage1_df["brand"] == "samsung"
].copy()

samsung_titles["samsung_model"] = samsung_titles["title_punct_normalized"].apply(
    lambda title: extract_brand_model(title, "samsung")
)

total = len(samsung_titles)
unknown = (samsung_titles["samsung_model"] == "unknown").sum()
known = total - unknown

print("Total Samsung ads:", total)
print("Models extracted:", known)
print("Unknown models:", unknown)
print(
    "Extraction coverage:",
    round(known / total * 100, 2) if total else 0,
    "%"
)

display(
    samsung_titles[
        ["title", "samsung_model"]
    ].head(20)
)

Total Samsung ads: 749
Models extracted: 740
Unknown models: 9
Extraction coverage: 98.8 %


,title,samsung_model
11,Samsung S24 Ultra زيرو 256GB تيتانيوم مدفوع ال...,samsung s24 ultra
14,Samsung S25 Ultra,samsung s25 ultra
19,Galaxy Z Flip 8,samsung z flip8
22,Galaxy Z Fold 8 ultra,samsung z fold8
25,Galaxy TriFold dual sim 512/16G Black جديد خال...,samsung z trifold
28,Galaxy S26 Ultra dual sim 256/12G Sky Blue Bl...,samsung s26 ultra
29,موبايل Samsung S23 Ultra مستعمل (256GB / 12GB ...,samsung s23 ultra
42,سامسونج ايه 52s,samsung a52s
49,سامسونج ايه 23,samsung a23
62,سامسونج اف 70 اي 5 جي,samsung f70


Review Unknown Samsung Models

In [ ]:
unknown_samsung = samsung_titles.loc[
    samsung_titles["samsung_model"] == "unknown",
    ["title"]
]

print("Unknown Samsung ads:", len(unknown_samsung))
display(unknown_samsung)

Unknown Samsung ads: 9


,title
353,سامسونج جالكسي Gt-55360
1247,فرصة عظيمة بسعر مغري لن يتكرر موبايل سامسون...
1789,بيع تليفون سامسونج جلاكسي الترا
1874,Samsung g4+0اقتصادي
2479,موبل سامسونج ِ30 جيد جدا للبيع
3369,موبايل سامسونج استعمال شهر كالجديد
4147,سامسونج8+لقطه
4331,سامسونج بأقل من ٣٠٠٠ جنيه
4436,سامسونج و أخرى وانواع مختلفه


Save Samsung Models to Stage 1 Dataset

In [ ]:
samsung_mask = stage1_df["brand"] == "samsung"

stage1_df.loc[samsung_mask, "samsung_model"] = (
    stage1_df.loc[samsung_mask, "title_punct_normalized"]
    .apply(lambda title: extract_brand_model(title, "samsung"))
)

print("Samsung model column added successfully.")
print("Samsung rows:", samsung_mask.sum())
print(
    "Samsung unknown:",
    (stage1_df.loc[samsung_mask, "samsung_model"] == "unknown").sum()
)

Samsung model column added successfully.
Samsung rows: 749
Samsung unknown: 9


Review Samsung Model Distribution

In [ ]:
display(
    stage1_df.loc[
        stage1_df["brand"] == "samsung",
        "samsung_model"
    ].value_counts().head(30)
)

,count
samsung_model,
samsung a56,43
samsung s25 ultra,35
samsung a17,28
samsung a36,23
samsung a07,22
samsung s24 ultra,21
samsung a55,20
samsung s22 ultra,20
samsung a52,20


Identify Samsung Parts and Accessories Listings

In [ ]:
parts_pattern = (
    r"شاشه|شاشة|قطع غيار|بورده|بوردة|بطاريه|بطارية|"
    r"جراب|اسكرينه|اسكرينة|شاحن|سماعه|سماعة|"
    r"screen|spare parts|battery|charger|case|cover"
)

samsung_parts = stage1_df.loc[
    (stage1_df["brand"] == "samsung")
    & stage1_df["title"].str.contains(
        parts_pattern, case=False, na=False, regex=True
    ),
    ["title", "samsung_model"]
]

print("Possible parts/accessories:", len(samsung_parts))
display(samsung_parts.head(30))

Possible parts/accessories: 19


,title,samsung_model
331,Samsung A32 128GB - سبق تغيير الشاشة بشاشة اصل...,samsung a32
1722,Galaxy M55 5Gمعاه شاحن45 الاورجينال وسكرين اما...,samsung m55
1791,سامسونج فولد٣ فاصل شاشه داخليه,samsung z fold3
1835,Samsung A5 78% battery charging port need fixing,samsung a5
1879,سامسونج ايه 15 رامات 8 مساحه 128 بدون كرتونه و...,samsung a15
2400,Samsung A30s متفتحش مغيرش حاجه بصمه شاشه طلقه ...,samsung a30s
2537,سامسونج A23 ٦ رام ١٢٨ مساحة تخزين بالشاحن الاصلي,samsung a23
2719,Samsung XCover 6 Pro,samsung xcover6 pro
2765,موبايل سامسونج A36 8 Ram- 256 GB بالعلبة والضم...,samsung a36
2768,سامسونج A52 حاله فوق الممتازه made in india با...,samsung a52


Mark Non-Phone Listings

In [ ]:
parts_only_pattern = (
    r"^\s*(?:شاشه|شاشة|بطاريه|بطارية|بورده|بوردة|"
    r"جراب|اسكرينه|اسكرينة|شاحن|سماعه|سماعة)\b"
    r"|محتاج شاشه|محتاج شاشة|مطلوب شاشه|مطلوب شاشة"
    r"|screen\s+(?:samsung|for\s+samsung)"
)

stage1_df["is_phone_listing"] = True

parts_mask = (
    (stage1_df["brand"] == "samsung")
    & stage1_df["title"].str.contains(
        parts_only_pattern, case=False, na=False, regex=True
    )
)

stage1_df.loc[parts_mask, "is_phone_listing"] = False

print("Non-phone listings marked:", (~stage1_df["is_phone_listing"]).sum())

display(
    stage1_df.loc[
        ~stage1_df["is_phone_listing"],
        ["title", "brand", "samsung_model"]
    ]
)

Non-phone listings marked: 1


,title,brand,samsung_model
3392,شاشة samsung a34 خلع اصلية,samsung,samsung a34


Review Brand Distribution in Training Data

In [ ]:
print("Brands in training data:")
display(stage1_df["brand"].value_counts(dropna=False))

Brands in training data:


,count
brand,
apple,2462
samsung,749
xiaomi,426
oppo,361
NaN,276
honor,224
realme,178
infinix,138
vivo,128


# Xiaomi

Some helper functions

In [ ]:
def explore_brand(brand, n=20, seed=10):
    brand_titles = stage1_df.loc[stage1_df["brand"] == brand].copy()
    print(f"Total {brand} ads:", len(brand_titles))
    display(
        brand_titles[["title", "title_punct_normalized"]].sample(
            n=min(n, len(brand_titles)), random_state=seed
        )
    )



def evaluate_brand(brand, show_rows=20, max_unknown=20):
    col = f"{brand}_model"
    mask = stage1_df["brand"] == brand

    # Extract models; unmatched titles return "unknown"
    results = stage1_df.loc[mask, "title"].apply(
        lambda title: extract_brand_model(title, brand)
    )

    # Save results
    stage1_df.loc[mask, col] = results

    # Evaluate extraction
    total = len(results)
    unknown = (results == "unknown").sum()
    known = total - unknown
    coverage = known / total * 100 if total else 0

    print(f"Brand: {brand}")
    print(f"Total ads: {total}")
    print(f"Models extracted: {known}")
    print(f"Unknown models: {unknown}")
    print(f"Extraction coverage: {coverage:.2f}%")




In [ ]:
explore_brand("xiaomi")

Total xiaomi ads: 426


,title,title_punct_normalized
3015,شاومي ريدمي ايه 3,شاومي ريدمي ايه 3
1824,شاومي ريدمي 13 سي,شاومي ريدمي 13 سي
1505,Xiaomi Redmi Note 14 Pro plus 5G للبيع,xiaomi redmi note 14 pro plus 5g للبيع
581,شاومي ريدمي نوت 15,شاومي ريدمي نوت 15
3372,شاومي ريدمي 9,شاومي ريدمي 9
2949,شاومي ريدمي 9 سى,شاومي ريدمي 9 سي
1066,شاومي ريدمي نوت 10,شاومي ريدمي نوت 10
323,Xiaomi Poco X3 (128/6),xiaomi poco x3 (128/6)
4881,redmi not 10 s للبدل,redmi not 10 s للبدل
4493,شاومي ريدمي ايه 3,شاومي ريدمي ايه 3


# Define Xiaomi Model Rules

In [ ]:

BRAND_CONFIG["xiaomi"] = {
    "letters": {
        "اكس": "x",
        "اف": "f",
        "ام": "m",
        "سي": "c",
        "كيه": "k",
        "ايه": "a",
        "تي": "t",
        "مي": "mi",
    },

    "aliases": {
        "ميكس": "mix",
        "تربو": "turbo",
        "بلاك شارك": "black shark",
    },

    "pre": [
        (r"xiaomi\s+مي\s+تن\s+تي", "xiaomi mi 10t"),
        (r"xiaomi\s+بوك\s+اكس\s+ثري\s+pro", "poco x3 pro"),
        (r"xiaomi\s+بوك\s+اكس\s+ثري", "poco x3"),
        (r"\bxiaomi\s+18\s*fold\b", "xiaomi 18 fold"),
        (r"\bxiaomi\s+fold\s+xiaomi\s+18\s*fold\b",
         "xiaomi fold xiaomi 18 fold"),
    ],

    "rules": [
        # Black Shark
        (
            r"\bblack\s*shark\s*(\d{1,2})(?:\s*(pro))?\b",
            "black shark {1} {2}"
        ),

        (
            r"\bredmi\s*note\s*(\d{1,2})"
            r"\s*(s|t|a)?"
            r"\s*(pro\s*plus|pro\s*max|pro|plus)?"
            r"\s*(5g)?\b",
            "redmi note {1}{2} {3} {4}"
        ),

        # Redmi K
        (
            r"\bredmi\s*k\s*(\d{2,3})"
            r"\s*(pro\s*max|pro|ultra|plus)?\b",
            "redmi k{1} {2}"
        ),

        # Redmi Turbo
        (
            r"\bredmi\s*turbo\s*(\d{1,2})\s*(pro|max)?\b",
            "redmi turbo {1} {2}"
        ),

        # Redmi A / C
        (
            r"\bredmi\s*([ac])\s*(\d{1,2})(y)?\b",
            "redmi {1}{2}{3}"
        ),

        # Redmi 12C / 13C / 12 5G / 13 5G وما شابه
        (
            r"\bredmi\s*(\d{1,2})\s*(c|a|t|s|x)"
            r"\s*(prime|pro|plus)?\b",
            "redmi {1}{2} {3}"
        ),

        (
            r"\bredmi\s*(\d{1,2})"
            r"\s*(prime|pro|plus)?\b",
            "redmi {1} {2}"
        ),

        # POCO X / F / M / C
        (
            r"\bpoco\s*([xfmc])\s*(\d{1,2})"
            r"\s*(pro|max|gt|ultra|plus)?\b",
            "poco {1}{2} {3}"
        ),

        (
            r"\bpoco\b.*?\b([xfmc])\s*(\d{1,2})"
            r"\s*(pro|max|gt|ultra|plus)?\b",
            "poco {1}{2} {3}"
        ),

        # Xiaomi Mix Fold / Flip
        (
            r"\bmix\s*(fold|flip)\s*(\d{1,2})?\b",
            "mix {1} {2}"
        ),

        # Xiaomi 18 Fold أو Xiaomi 18fold
        (
            r"\bxiaomi\s*(\d{1,2})\s*fold\b",
            "{1} fold"
        ),

        # Mi A
        (
            r"\bmi\s*a\s*(\d{1,2})\b",
            "mi a{1}"
        ),

        # Mi Note
        (
            r"\bmi\s*note\s*(\d{1,2})\s*(pro|lite)?\b",
            "mi note {1} {2}"
        ),

        # Mi / Mi T
        (
            r"\bmi\s*(\d{1,2}t?)\s*(pro|lite|ultra)?\b",
            "mi {1} {2}"
        ),

        # Xiaomi 12C -> Redmi 12C
        (
            r"\bxiaomi\s*(\d{1,2})\s*c\b",
            "redmi {1}c"
        ),

        # Xiaomi A3 -> Redmi A3
        (
            r"\bxiaomi\s*a\s*(\d{1,2})\b",
            "redmi a{1}"
        ),

        (
            r"\bxiaomi\s*(\d{1,2}t?)"
            r"\s*(pro|lite|ultra|plus)?\b",
            "{1} {2}"
        ),

        # Ultra 15 -> 15 ultra
        (
            r"\bultra\s*(\d{2})\b",
            "{1} ultra"
        ),

        (
            r"\bc\s*(\d{2})\b",
            "redmi c{1}"
        ),

        (
            r"\bnote\s*(\d{1,2})"
            r"\s*(s|t)?"
            r"\s*(pro\s*plus|pro\s*max|pro|plus)?"
            r"\s*(5g)?\b",
            "redmi note {1}{2} {3} {4}"
        ),
    ],
}


In [ ]:
evaluate_brand("xiaomi")

Brand: xiaomi
Total ads: 426
Models extracted: 398
Unknown models: 28
Extraction coverage: 93.43%


In [ ]:
stage1_df.loc[
    stage1_df["xiaomi_model"].eq("unknown"),
    ["title", "title_normalized", "xiaomi_model"]
].head(69)

,title,title_normalized,xiaomi_model
217,Poco بوكو,poco بوكو,unknown
412,شاومي 111T,شاومي 111t,unknown
585,شاومي موديل اخر,شاومي موديل اخر,unknown
727,شاومي موديل اخر,شاومي موديل اخر,unknown
770,شاومي موديل اخر,شاومي موديل اخر,unknown
1032,شاومي موديل اخر,شاومي موديل اخر,unknown
1391,شاومي موديل اخر,شاومي موديل اخر,unknown
1587,شاومي موديل اخر,شاومي موديل اخر,unknown
1768,شاومي ريد مى,شاومي ريد مي,unknown
1787,شاومي موديل اخر,شاومي موديل اخر,unknown


# Oppo

In [ ]:
explore_brand("oppo")

Total oppo ads: 361


,title,title_punct_normalized
2934,اوبو رينو 15 5g,اوبو رينو 15 5g
1432,Oppo Find N5,oppo find n5
1692,oppo a3s,oppo a3s
67,oppo a5s,oppo a5s
1854,اوبو رينو 6,اوبو رينو 6
769,اوبو رينو14 اف,اوبو رينو14 اف
2465,اوبو رينو 11 اف,اوبو رينو 11 اف
1919,اوبو موديل اخر,اوبو موديل اخر
1222,OPPO Reno12 F,oppo reno12 f
2706,جهاز oppo a3,جهاز oppo a3


In [ ]:

BRAND_CONFIG["oppo"] = {
    "letters": {
        "ايه": "a",
        "اي": "a",
        "اية": "a",
        "اف": "f",
        "ار": "r",
        "تي": "t",
        "اكس": "x",
    },

    "aliases": {
        "opoo": "oppo",
        "oppooo": "oppo",
        "فايف جي": "5g",
        "فايند": "find",
    },

    "pre": [
        (r"\bfind\s+ان\s*(?=\d)", "find n "),

        # Turbo: تربو -> turbo
        (
            r"(?<![\u0600-\u06FF])تربو(?![\u0600-\u06FF])",
            "turbo"
        ),

        (r"\ba\s*6\s+اي\b", "a6i"),

        # A6 Pro
        (r"\ba\s*6\s+pro\b", "a6pro"),
    ],

    "rules": [
        (r"\ba\s*6a\b", "a6i"),

        # Find N series
        (
            r"\bfind\s*n\s*(\d{1,2})\b",
            "find n{1}"
        ),

        # Find X series
        (
            r"\bfind\s*x\s*(\d{1,2})\s*(pro|plus|ultra)?\b",
            "find x{1} {2}"
        ),

        # K series: K13 Turbo Pro 5G
        (
            r"\bk\s*(\d{1,2})\s*turbo\s*(pro)?\s*(5g)?\b",
            "k{1} turbo {2} {3}"
        ),

        # Reno series: Reno 7, Reno 8T, Reno 15F 5G
        (
            r"\breno\s*(\d{1,2})\s*(t|f|z)?\s*"
            r"(pro\s*plus|pro|plus)?\s*(5g)?\b",
            "reno {1}{2} {3} {4}"
        ),

        # A series: A60, A5i, A6 Pro, A6X, A1K
        (
            r"\ba\s*(\d{1,3})\s*"
            r"(pro\s*plus|pro|plus|x|k|s|i)?\s*(5g)?\b",
            "a{1}{2} {3}"
        ),

        # F series: F5, F1s
        (
            r"\bf\s*(\d{1,2})\s*(s|x|pro)?\b",
            "f{1}{2}"
        ),

        # R series: R15 Pro
        (
            r"\br\s*(\d{1,2})\s*(pro|plus)?\b",
            "r{1} {2}"
        ),
    ]
}





In [ ]:
evaluate_brand("oppo")

Brand: oppo
Total ads: 361
Models extracted: 320
Unknown models: 41
Extraction coverage: 88.64%


In [ ]:
stage1_df.loc[
    stage1_df["oppo_model"].eq("unknown"),
    ["title", "title_normalized", "oppo_model"]
].head(69)

,title,title_normalized,oppo_model
348,اوبو موديل اخر,اوبو موديل اخر,unknown
374,كوبي نسخه اولي اصدار اماراتي OPPO 16 وصــــــ...,كوبي نسخه اولي اصدار اماراتي oppo 16 وصــــــ...,unknown
747,اوبو موديل اخر,اوبو موديل اخر,unknown
917,اوبو موديل اخر,اوبو موديل اخر,unknown
984,اوبو 16f,اوبو 16f,unknown
1074,اوبو موديل اخر,اوبو موديل اخر,unknown
1223,OPPO Other Model,oppo other model,unknown
1339,اوبو فايند إن,اوبو فايند ان,unknown
1361,اوبو موديل اخر,اوبو موديل اخر,unknown
1383,اوبو موديل اخر,اوبو موديل اخر,unknown


# honor

In [ ]:
explore_brand("honor")

Total honor ads: 224


,title,title_punct_normalized
2293,هونر 600,هونر 600
762,هونر اكس 9 سى,هونر اكس 9 سي
2054,هونر ماجيك 7,هونر ماجيك 7
1906,هونر x8d,هونر x8d
2069,Honor 600 للبدل او البيع,honor 600 للبدل او البيع
3869,لو عندك موبايل هونر عاوز تبيعو مستعمل او جديد...,لو عندك موبايل هونر عاوز تبيعو مستعمل او جديد ...
2745,Honor X7c,honor x7c
1779,Honor 600,honor 600
1984,Honor Magic8 Pro,honor magic8 pro
4614,هونر 400 برو,هونر 400 برو


In [ ]:

BRAND_CONFIG["honor"] = {
    "letters": {
        "اكس": "x",
        "سي": "c",
        "سى": "c",
        "سيه": "c",
        "سيي": "c",
        "بي": "b",
        "بى": "b",
        "ايه": "a",
        "اي": "i",
    },

    "aliases": {
        "honour": "honor",
        "هونور": "honor",
        "اونر": "honor",
    },

    "pre": [
        # Arabic X: اكس 8 -> x 8
        (r"(?<![\u0600-\u06FF])اكس(?![\u0600-\u06FF])", "x"),

        # Arabic Magic V: ماجيك في 3 -> magic v 3
        (r"ماجيك\s+في(?=\s*\d)", "magic v"),

        # Correct likely typo only in the specific model context
        (r"\bhonor\s+wen\s+rt\b", "honor win rt"),
        (r"\bhonor\s+wen(?=\s+5g\b)", "honor win"),
    ],

    "rules": [
        # Honor X series: X6, X6b, X7a, X8c, X9c
        (r"\bx\s*(\d{1,2})\s*(c|b|a|i|d|pro|plus)?\b", "x{1}{2}"),

        # Arabic reversed format: 6 X -> X6
        (r"\b(\d{1,2})\s*x\s*(c|b|a)?\b", "x{1}{2}"),

        # Magic V series with a version number
        (r"\bmagic\s+v\s*(\d{1,2})\s*(pro|plus)?\b", "magic v{1} {2}"),

        # Magic V without a version number
        (r"\bmagic\s+v\b", "magic v"),

        # Magic numeric series: Magic 5 Pro, Magic 8 Pro
        (r"\bmagic\s*(\d{1,2})\s*(pro\s*plus|pro|plus|lite)?\b", "magic {1} {2}"),

        # Honor numeric series: Honor 200 Pro, Honor 600
        (r"\bhonor\s+(\d{2,3})\s*(pro\s*plus|pro|plus|lite)?\b", "{1} {2}"),

        # Honor Win RT / Win
        (r"\bwin\s+rt\b", "win rt"),
        (r"\bwin\b", "win"),

        # Honor Robot Phone / Robot
        (r"\brobot(?:\s+phone)?\b", "robot"),

        # Honor Play
        (r"\bplay\b", "play"),

        # Honor GT
        (r"\bgt\b", "gt"),

        # Honor 8 Pro / 8 Plus / 8 Lite
        (r"\bhonor\s+(\d{1,2})\s+(pro|plus|lite)\b", "{1} {2}"),

        # Honor 8X / 8C
        (r"\bhonor\s+(\d{1,2})\s*(x|c)\b", "{1}{2}"),
    ],
}


In [ ]:
evaluate_brand("honor")

Brand: honor
Total ads: 224
Models extracted: 209
Unknown models: 15
Extraction coverage: 93.30%


In [ ]:
stage1_df.loc[
    stage1_df["honor_model"].eq("unknown"),
    ["title", "title_normalized", "honor_model"]
].head(69)

,title,title_normalized,honor_model
1387,هونر موديل اخر,هونر موديل اخر,unknown
1763,هونر موديل اخر,هونر موديل اخر,unknown
1845,هونر ماجيك في 3 ( فولد ينطوي),هونر ماجيك في 3 ( فولد ينطوي),unknown
2439,هونر موديل اخر,هونر موديل اخر,unknown
2475,Honor 7Xe,honor 7xe,unknown
2487,هونر موديل اخر,هونر موديل اخر,unknown
2790,هونر ماجيك في 2,هونر ماجيك في 2,unknown
3693,هونر موديل اخر,هونر موديل اخر,unknown
3865,هونر موديل اخر,هونر موديل اخر,unknown
3869,لو عندك موبايل هونر عاوز تبيعو مستعمل او جديد...,لو عندك موبايل هونر عاوز تبيعو مستعمل او جديد...,unknown


# Realme

In [ ]:
explore_brand("realme")

Total realme ads: 178


,title,title_punct_normalized
1764,ريلمي 12 برو,ريلمي 12 برو
2940,Realme 14 Pro plus,realme 14 pro plus
860,ريلمى جى تى 7 برو- realme 7 pro,ريلمي جي تي 7 برو realme 7 pro
3389,Realme C15,realme c15
1553,ريلمى سى 71,ريلمي سي 71
2806,Realme 11 Pro,realme 11 pro
3399,ريلمى 14 برو,ريلمي 14 برو
1163,ريلمى جي تي 6,ريلمي جي تي 6
5034,ريلمى بى 1 برو,ريلمي بي 1 برو
465,ريلمى سى 71,ريلمي سي 71


In [ ]:

BRAND_CONFIG["realme"] = {
    "letters": {
        "سي": "c",
        "سى": "c",
        "بي": "p",
        "بى": "p",
        "س": "c",
    },

    "aliases": {
        "ريلمى": "realme",
        "نارزو": "narzo",
        "نوت": "note",
    },

    "pre": [
        # Arabic model names: specific phrases must come first
        (r"جي\s*تي\s*نيو", "gt neo"),
        (r"جي\s*تي", "gt"),
        (r"نوت(?=\s*\d)", "note"),
        (r"في(?=\s*\d)", "v"),

        # Fix common spelling variations
        (r"\bnot(?=\s*\d)", "note"),
        (r"\bgt(?=\d)", "gt "),
        (r"\bc(?=\d)", "c "),
        (r"\bp(?=\d)", "p "),
        (r"\ba(?=\d)", "a "),
        (r"\bproه\b", "pro"),
        (r"بروه\b", "pro"),
    ],

    "rules": [
        # GT Neo series — place before general GT rules
        (r"\bgt\s*neo\s*(\d{1,2})\s*(t)?\b", "gt neo {1}{2}"),

        # GT Master Edition
        (r"\bgt\s*master\s*edition\b", "gt master edition"),

        # GT Pro series
        (r"\bgt\s*(\d{1,2})\s*pro\b", "gt {1} pro"),

        # GT T series: GT 6T -> gt 6t
        (r"\bgt\s*(\d{1,2})\s*t\b", "gt {1}t"),

        # Standard GT series
        (r"\bgt\s*(\d{1,2})\b", "gt {1}"),

        # Note series
        (r"\bnote\s*(\d{2,3})\s*(x)?\b", "note {1}{2}"),

        # C series: C21Y, C100i, C3
        (r"\bc\s*(\d{1,3})\s*([a-z])?\b", "c{1}{2}"),

        # P series
        (r"\bp\s*(\d{1,2})\s*(ultra|pro\s*plus|pro|plus)?\b", "p{1} {2}"),

        # Narzo series
        (r"\bnarzo\s*(\d{1,3})\s*(pro|plus|x)?\b", "narzo {1} {2}"),

        # V series
        (r"\bv\s*(\d{1,2})\s*(t|i)?\b", "v{1}{2}"),

        # Numbered series with suffixes: Realme 9i, 14t, etc.
        (r"\brealme\s+(\d{1,2})\s*(pro\s*plus|pro|plus)\b", "{1} {2}"),
        (r"\brealme\s+(\d{1,2})\s*(i|t|x)\b", "{1}{2}"),
        (r"\brealme\s+(\d{1,2})\b", "{1}"),

        # A series
        (r"\ba\s*(\d{1,2})\s*(pro|plus)?\b", "a{1} {2}"),
    ],
}


In [ ]:
evaluate_brand("realme")

Brand: realme
Total ads: 178
Models extracted: 175
Unknown models: 3
Extraction coverage: 98.31%


In [ ]:
stage1_df.loc[
    stage1_df["realme_model"].eq("unknown"),
    ["title", "title_normalized", "realme_model"]
].head(69)

,title,title_normalized,realme_model
806,ريلمي,ريلمي,unknown
1511,ريلمي,ريلمي,unknown
2351,ريلمى موديل i9 G5,ريلمي موديل i9 g5,unknown


# infinix

In [ ]:
explore_brand("infinix")

Total infinix ads: 138


,title,title_punct_normalized
2771,إنفنيكس نوت 8i,انفنيكس نوت 8i
4999,إنفنيكس هوت 60 برو,انفنيكس هوت 60 برو
1171,إنفنيكس نوت ايدج,انفنيكس نوت ايدج
3921,Infinix Hot 11,infinix hot 11
4889,infinix gt 30 pro \ انفينكس جي تي 30 برو,infinix gt 30 pro \ انفينكس جي تي 30 برو
2615,إنفنيكس موديل اخر,انفنيكس موديل اخر
3657,إنفنيكس جي تي 30 برو,انفنيكس جي تي 30 برو
2222,إنفنيكس نوت 50,انفنيكس نوت 50
3122,Infinix Hot 60i – 12GB RAM / 128GB,infinix hot 60i – 12gb ram / 128gb
3848,إنفنيكس هوت 11 بلاي,انفنيكس هوت 11 بلاي


In [ ]:

BRAND_CONFIG["infinix"] = {
    "letters": {},

    "aliases": {
        "انفنيكس": "infinix",
        "انفنكس": "infinix",
        "انفينكس": "infinix",
        "إنفنيكس": "infinix",
        "سمارت": "smart",
        "سمار": "smart",
        "هوت": "hot",
        "نوت": "note",
        "زيرو": "zero",
        "ايدج": "edge",
        "برو": "pro",
        "بلاي": "play",
    },

    "pre": [
        # Arabic series names
        (r"جي\s*تي", "gt"),
        (r"اس(?=\s*\d)", "s"),
        (r"سمارت\s*زيرو", "smart zero"),

        # Smart 6 اي -> Smart 6i
        (r"\bsmart\s*(\d{1,2})\s*اي\b", r"smart \1i"),

        # Join brand and model if written together
        (r"\binfinix(?=[a-z])", "infinix "),

        # Separate series from numbers
        (r"\b(gt|smart|note|hot|zero|s)\s*(?=\d)", r"\1 "),

        # Separate suffixes from numbers
        (r"(?<=\d)(pro|plus|play|ultra|lite)\b", r" \1"),
    ],

    "rules": [
        # GT series
        (r"\bgt\s*(\d{1,2})\s*pro\b", "gt {1} pro"),
        (r"\bgt\s*(\d{1,2})\s*plus\b", "gt {1} plus"),
        (r"\bgt\s*(\d{1,2})\b", "gt {1}"),

        # Smart Zero
        (r"\bsmart\s+zero\b", "smart zero"),

        # Smart series
        (r"\bsmart\s*(\d{1,2})\s*i\b", "smart {1}i"),
        (r"\bsmart\s*(\d{1,2})\b", "smart {1}"),

        # Zero series: Zero 8i, Zero 30, etc.
        (r"\bzero\s*(\d{1,2})\s*(i|x|pro)?\b", "zero {1}{2}"),

        # Note Edge
        (r"\bnote\s*edge\b", "note edge"),

        # Note series: Note 50s, Note 60
        (r"\bnote\s*(\d{1,3})\s*(s|pro|plus|x|i)?\b", "note {1}{2}"),

        # Hot series with letter suffix: Hot 60i
        (r"\bhot\s*(\d{1,3})\s*(i|s|x)\b", "hot {1}{2}"),

        # Hot series with word suffix: Hot 11 Play, Hot 60 Pro
        (
            r"\bhot\s*(\d{1,3})\s*(play|pro\s*plus|pro|plus|ultra|lite)?\b",
            "hot {1} {2}"
        ),

        # S series: S4, S5
        (r"\bs\s*(\d{1,2})\s*(pro|plus|lite)?\b", "s{1} {2}"),
    ],
}


In [ ]:
evaluate_brand("infinix")

Brand: infinix
Total ads: 138
Models extracted: 118
Unknown models: 20
Extraction coverage: 85.51%


In [ ]:
stage1_df.loc[
    stage1_df["infinix_model"].eq("unknown"),
    ["title", "title_normalized", "infinix_model"]
].head(69)

,title,title_normalized,infinix_model
46,تليفون انفينكس للبيع,تليفون انفينكس للبيع,unknown
398,إنفنيكس موديل اخر,انفنيكس موديل اخر,unknown
432,إنفنيكس موديل اخر,انفنيكس موديل اخر,unknown
560,موبايل مستعمل انفنيكس سمارت,موبايل مستعمل انفنيكس سمارت,unknown
746,إنفنيكس موديل اخر,انفنيكس موديل اخر,unknown
889,إنفنيكس موديل اخر,انفنيكس موديل اخر,unknown
909,Infinix Smart,infinix smart,unknown
1064,إنفنيكس موديل اخر,انفنيكس موديل اخر,unknown
1102,إنفنيكس موديل اخر,انفنيكس موديل اخر,unknown
1557,إنفنيكس موديل اخر,انفنيكس موديل اخر,unknown


# Vivo

In [ ]:
explore_brand("vivo")

Total vivo ads: 128


,title,title_punct_normalized
4400,فيفو واي ٣١ دي,فيفو واي 31 دي
3988,فيفو فى 50,فيفو في 50
1958,فيفو واي 28,فيفو واي 28
5100,VIVO V 60 LITE 5G,vivo v 60 lite 5g
4685,فيفو اس 19 برو,فيفو اس 19 برو
2715,فيفو فى 50,فيفو في 50
2478,فيفو واى 29,فيفو واي 29
3056,vivo v50 lite 5g,vivo v50 lite 5g
2111,فيفو واى 12 اس,فيفو واي 12 اس
1893,ViVO Y29 مساحة 128/8 بحالة الزيرو- موبايل فيفو,vivo y29 مساحة 128/8 بحالة الزيرو موبايل فيفو


In [ ]:

BRAND_CONFIG["vivo"] = {
    "letters": {},

    "aliases": {
        "فيفو": "vivo",
        "اكس": "x",
        "إكس": "x",
        "فولد": "fold",
        "اف اي": "fe",
        "إف إي": "fe",
        "واي": "y",
        "واى": "y",
        "اس": "s",
        "إس": "s",
        "برو": "pro",
        "لايت": "lite",
        "الترا": "ultra",
        "نيو": "neo",
    },

    "pre": [
        # Arabic V-series notation: في 70 -> v 70
        (r"في(?=\s*\d)", "v"),

        # Arabic iQOO spelling variants
        (r"اي\s*كيو\s*او(?:و)?", "iqoo"),

        # Arabic model suffixes
        (r"اف\s*اي\b", "fe"),
        (r"اي(?=\s|$)", "e"),

        # Smart spacing for concatenated brand and model
        (r"\bvivo(?=[a-z])", "vivo "),

        # X Fold 3 Pro and similar
        (r"\bx\s*fold\s*(?=\d)", "x fold "),

        # Separate series from model number
        (r"\b(x|y|v|s)\s*(?=\d)", r"\1 "),

        # Separate suffixes when written directly after digits
        (r"(?<=\d)(pro|plus|ultra|lite|fe)\b", r" \1"),
    ],

    "rules": [
        # iQOO Neo series
        (
            r"\biqoo\s*neo\s*(\d{1,3})\s*(pro\s*plus|pro|plus|ultra)?\b",
            "iqoo neo {1} {2}"
        ),

        # X Fold series: X Fold 3 Pro
        (
            r"\bx\s*fold\s*(\d{1,2})\s*(pro|plus|ultra)?\b",
            "x fold {1} {2}"
        ),

        # X series with FE suffix: X300 FE
        (
            r"\bx\s*(\d{2,3})\s*(fe|ultra|pro|plus|lite|e)?\b",
            "x{1} {2}"
        ),

        # V series with suffix: V23e, V27e, V29e
        (
            r"\bv\s*(\d{1,3})\s*(e|s|d)\s*(pro|plus|lite|ultra)?\b",
            "v{1}{2} {3}"
        ),

        # V series with Pro / Lite / Ultra
        (
            r"\bv\s*(\d{1,3})\s*(pro|plus|lite|ultra)\b",
            "v{1} {2}"
        ),

        # V series without suffix
        (
            r"\bv\s*(\d{1,3})\b",
            "v{1}"
        ),

        # Titles like "Vivo 40 Lite" or "Vivo 50 Lite"
        (
            r"\bvivo\s+(\d{1,3})\s*lite\b",
            "v{1} lite"
        ),

        # S series with suffix
        (
            r"\bs\s*(\d{1,3})\s*(pro|plus|lite|ultra)?\b",
            "s{1} {2}"
        ),

        # Y series with suffixes: Y11D, Y31D, Y20s, Y17s
        (
            r"\by\s*(\d{1,3})\s*(s|d|e)\s*(pro|plus|lite|ultra)?\b",
            "y{1}{2} {3}"
        ),

        # Y series with Pro / Lite / Plus
        (
            r"\by\s*(\d{1,3})\s*(pro|plus|lite|ultra)\b",
            "y{1} {2}"
        ),

        # Standard Y series
        (
            r"\by\s*(\d{1,3})\b",
            "y{1}"
        ),
    ],
}


In [ ]:
evaluate_brand("vivo")

Brand: vivo
Total ads: 128
Models extracted: 123
Unknown models: 5
Extraction coverage: 96.09%


In [ ]:
stage1_df.loc[
    stage1_df["vivo_model"].eq("unknown"),
    ["title", "title_normalized", "vivo_model"]
].head(69)

,title,title_normalized,vivo_model
346,فيفو ٢٠١٥,فيفو 2015,unknown
704,Vivo Other Model,vivo other model,unknown
2402,فيفووى11دى,فيفووي11دي,unknown
3298,فيفو موديل اخر للبيع او البدل بأعلي مع دفع الفرق,فيفو موديل اخر للبيع او البدل باعلي مع دفع الفرق,unknown
3927,فيفو 19 جهاز زيروعجلة سرعات زيرو 6سرعات جميل...,فيفو 19 جهاز زيروعجلة سرعات زيرو 6سرعات جميل...,unknown


In [ ]:
explore_brand("huawei")

Total huawei ads: 99


,title,title_punct_normalized
1000,Huawei Nova 15 MAX,huawei nova 15 max
658,Huawei Y5p ٢شريحة و ٢ gb ram ٣٢ rom لون Navy blue,huawei y5p 2شريحة و 2 gb ram 32 rom لون navy blue
1781,huawei هواوي,huawei هواوي
1624,هواوي موديل اخر,هواوي موديل اخر
2992,هواوي نوفا 7 se نسخه5,هواوي نوفا 7 se نسخه5
255,هاتف هواوي واي,هاتف هواوي واي
4225,هواوي ميت 40 برو,هواوي ميت 40 برو
1715,هواوي نوفا 1٥ ماكس,هواوي نوفا 15 ماكس
1625,موبايلين Huawei للبيع للتصليح أو قطع الغيار,موبايلين huawei للبيع للتصليح او قطع الغيار
3227,Huawei Other Model,huawei other model


In [ ]:
BRAND_CONFIG["huawei"] = {
    "letters": {},

    "aliases": {
        "هواوي": "huawei",
        "هواوى": "huawei",
        "هواويه": "huawei",

        "ميت": "mate",
        "نوفا": "nova",
        "بورا": "pura",

        "برايم": "prime",
        "برو": "pro",
        "لايت": "lite",
        "بلس": "plus",
        "الترا": "ultra",
        "التميت": "ultimate",

        "بي": "p",
        "بى": "p",
        "واي": "y",
        "واى": "y",
        "اكس": "x",
        "إكس": "x",
        "سي": "se",
    },

    "pre": [
        (r"ميت\s*(?=[a-z0-9])", "mate "),
        (r"نوفا\s*(?=\d)", "nova "),
        (r"بورا\s*(?=\d)", "pura "),

        (r"\bmat(?=\s*\d)", "mate "),
        (r"\bhuawei(?=[a-z])", "huawei "),

        (r"\b(y|p|nova|mate|pura)\s*(?=\d)", r"\1 "),
        (r"\bmate\s*x\s*t\s*(?=\d)", "mate xt "),
        (r"\bmate\s*x\s*(?=\d)", "mate x "),

        (r"\by\s*(\d{1,2})([a-z])\b", r"y \1\2"),

        (r"\by\s*(9)\s*(2019)\b", r"y \1 \2"),

        (r"(?<=\d)(pro|max|plus|ultra|lite|prime|ultimate)\b", r" \1"),
    ],

    "rules": [
        # -------------------------
        # Huawei Pura series
        # -------------------------
        (r"\bpura\s*(\d{2})\s*s\s*pro\s*max\b",
         "pura {1}s pro max"),

        (r"\bpura\s*(\d{2})\s*pro\s*max\b",
         "pura {1} pro max"),

        (r"\bpura\s*(\d{2})\s*(ultra|pro|plus|lite|max)\b",
         "pura {1} {2}"),

        (r"\bpura\s*(\d{2})\b",
         "pura {1}"),

        # -------------------------
        # Huawei Mate XT series
        # -------------------------
        (r"\bmate\s*xt\s*(\d{1,2})\s*(ultimate|pro|plus)?\b",
         "mate xt{1} {2}"),

        # -------------------------
        # Huawei Mate X series
        # -------------------------
        (r"\bmate\s*x\s*(\d{1,2})\s*(ultimate|pro|plus)?\b",
         "mate x{1} {2}"),

        # -------------------------
        # Huawei Mate series
        # -------------------------
        (r"\bmate\s*(\d{1,2})\s*(pro|plus|lite|x)?\b",
         "mate {1} {2}"),

        # -------------------------
        # Huawei Nova series
        # -------------------------
        (r"\bnova\s*(\d{1,2})\s*(se|i|pro|plus|lite|max)?\b",
         "nova {1}{2}"),

        # -------------------------
        # Huawei P series
        # -------------------------
        (r"\bp\s*(\d{1,2})\s*(pro|plus|lite|max)?\b",
         "p{1} {2}"),

        # -------------------------
        # Huawei Y series
        # -------------------------
        (r"\by\s*(7)\s*a\b",
         "y{1}a"),

        (r"\by\s*(6)\s*p\b",
         "y{1}p"),

        (r"\by\s*(\d{1,2})\s*(prime)\s*(plus)\b",
         "y{1} prime plus"),

        (r"\by\s*(\d{1,2})\s*(prime)\s*(2019)\b",
         "y{1} prime {3}"),

        (r"\by\s*(\d{1,2})\s*(2019|2020)\b",
         "y{1} {2}"),

        (r"\by\s*(\d{1,2})\s*(prime|pro|plus|lite|p|s)?\b",
         "y{1} {2}"),
    ],
}

In [ ]:
evaluate_brand("huawei")

Brand: huawei
Total ads: 99
Models extracted: 72
Unknown models: 27
Extraction coverage: 72.73%


In [ ]:
stage1_df.loc[
    stage1_df["huawei_model"].eq("unknown"),
    ["title", "title_normalized", "huawei_model"]
].head(69)

,title,title_normalized,huawei_model
202,هواوي موديل اخر,هواوي موديل اخر,unknown
255,هاتف هواوي واي,هاتف هواوي واي,unknown
257,هواوى نوفا ماكس استعمال 10 ايام بالفاتورة والض...,هواوي نوفا ماكس استعمال 10 ايام بالفاتورة والض...,unknown
808,تلفون هواوي للبيع شغال,تلفون هواوي للبيع شغال,unknown
990,هواوي موديل اخر,هواوي موديل اخر,unknown
1217,هواوي موديل اخر,هواوي موديل اخر,unknown
1312,هواوي ميت اكس تي التميت,هواوي ميت اكس تي التميت,unknown
1343,Huawei Other Model,huawei other model,unknown
1451,هواوي موديل اخر,هواوي موديل اخر,unknown
1488,هواوي موديل اخر,هواوي موديل اخر,unknown


# motorola

In [ ]:
explore_brand("motorola")

Total motorola ads: 26


,title,title_punct_normalized
3710,Motorola Signature,motorola signature
997,موتورولا موتو جي 35,موتورولا موتو جي 35
2056,هاتف Motorola Moto G5 Plus بحاله ممتازه,هاتف motorola moto g5 plus بحاله ممتازه
1087,Motorola Edge 60 fusion,motorola edge 60 fusion
139,موتورولا ايدج 2023+,موتورولا ايدج 2023+
209,موتورولا موتو جى 85,موتورولا موتو جي 85
1594,موتورولا موتو جي 15,موتورولا موتو جي 15
4706,موتورولا موتو جي,موتورولا موتو جي
1751,موتورولا موتو جى 85 كسر كسر الزيرو تم تنزيل ال...,موتورولا موتو جي 85 كسر كسر الزيرو تم تنزيل ال...
1677,موتورولا موتو جى 85,موتورولا موتو جي 85


In [ ]:
BRAND_CONFIG["motorola"] = {
    "letters": {},

    "aliases": {
        # Brand
        "موتورولا": "motorola",
        "موتورله": "motorola",

        # Moto G
        "موتو": "moto",
        "جي": "g",
        "جى": "g",

        # Edge
        "ايدج": "edge",
        "إيدج": "edge",

        # Razr
        "رازر": "razr",
        "ريزر": "razr",

        # Stylus
        "ستايلس": "stylus",
        "ستايلوس": "stylus",

        # Other model words
        "فيوجن": "fusion",
        "الترا": "ultra",
        "زيروو": "zero",
    },

    "pre": [
        # Brand normalization
        (r"\bmotorola(?=[a-z0-9])", "motorola "),

        # Moto G series
        (r"\bmoto\s*g\s*(?=\d)", "moto g "),

        # Edge series
        (r"\bedge\s*(?=\d)", "edge "),

        # Razr series
        (r"\brazr\s*(?=\d)", "razr "),

        # Stylus series
        (r"\bstylus\s*(?=\d)", "stylus "),

        # G04, G24, etc.
        (r"\bg\s*(?=\d)", "g "),

        # Separate suffixes attached to digits
        (r"(?<=\d)(ultra|fusion|stylus)\b", r" \1"),
    ],

    "rules": [
        # -------------------------
        # Motorola Edge
        # -------------------------

        # Reversed title: 50 Edge Fusion
        (r"\b(\d{2})\s*edge\s*fusion\b",
         "edge {1} fusion"),

        # Edge 50 Fusion, Edge 60 Fusion
        (r"\bedge\s*(\d{2})\s*fusion\b",
         "edge {1} fusion"),

        # Edge models with suffix
        (r"\bedge\s*(\d{2})\s*(ultra|pro|neo|fusion|plus)\b",
         "edge {1} {2}"),

        # Edge models without suffix
        (r"\bedge\s*(\d{2})\b",
         "edge {1}"),

        # Edge 2023+
        (r"\bedge\s*(2023)\s*\+",
         "edge {1} plus"),

        (r"\bedge\s*(\d{4})\b",
         "edge {1}"),

        # -------------------------
        # Motorola Razr
        # -------------------------
        (r"\brazr\s*(\d{2})\s*(ultra|plus)?\b",
         "razr {1} {2}"),

        # -------------------------
        # Motorola Stylus
        # -------------------------
        (r"\bstylus\s*5g\b",
         "stylus 5g"),

        (r"\bstylus\s*(\d{1,2})\s*(5g)?\b",
         "stylus {1} {2}"),

        # -------------------------
        # Motorola Moto G
        # -------------------------

        # Moto G5 Plus, Moto G85, Moto G56, etc.
        (r"\bmoto\s*g\s*(\d{1,3})\s*(plus|power|play|pro)?\b",
         "moto g{1} {2}"),

        # G04, G24, G85, etc. without Moto
        (r"\bg\s*(\d{2,3})\s*(plus|power|play|pro)?\b",
         "g{1} {2}"),

        # -------------------------
        # Motorola Signature
        # -------------------------
        (r"\bsignature\b",
         "signature"),
    ],
}

In [ ]:
evaluate_brand("motorola")

Brand: motorola
Total ads: 26
Models extracted: 23
Unknown models: 3
Extraction coverage: 88.46%


In [ ]:
stage1_df.loc[
    stage1_df["motorola_model"].eq("unknown"),
    ["title", "title_normalized", "motorola_model"]
].head(69)

,title,title_normalized,motorola_model
4491,Motorola 60 age fugen,motorola 60 age fugen,unknown
4706,موتورولا موتو جي,موتورولا موتو جي,unknown
4918,موتورولا موديل اخر,موتورولا موديل اخر,unknown


# oneplus

In [ ]:
explore_brand("oneplus")

Total oneplus ads: 17


,title,title_punct_normalized
1380,وان بلس ايس 6,وان بلس ايس 6
723,ون بلس نورد ٤ مساحة ٥١٢ جيجا,ون بلس نورد 4 مساحة 512 جيجا
3977,وان بلس 12 ار one +12r,وان بلس 12 ار one +12r
1769,Oneplus Ace 5 256g 12g ram زيروووو,oneplus ace 5 256g 12g ram زيروووو
1510,OnePlus 11وان بلس ١١ حالة ممتازة,oneplus 11وان بلس 11 حالة ممتازة
1829,تليفون رائد بمعالج سناب دراجون الجيل الثامن On...,تليفون رائد بمعالج سناب دراجون الجيل الثامن on...
2001,Oneplus 10 pro,oneplus 10 pro
697,OnePlus 12r علبه انجليزي بحالته وجميع مشتملاته,oneplus 12r علبه انجليزي بحالته وجميع مشتملاته
2415,Oneplus 7tpro mclaren edition,oneplus 7tpro mclaren edition
4866,One Plus Nord 4,one plus nord 4


In [ ]:
BRAND_CONFIG["oneplus"] = {
    "letters": {},

    "aliases": {
        # Brand names in Arabic
        "وان بلس": "oneplus",
        "ون بلس": "oneplus",
        "وانبلس": "oneplus",
        "ونبلس": "oneplus",

        # Series names
        "ايس": "ace",
        "آيس": "ace",
        "نورد": "nord",
        "نورد سي": "nord ce",
        "ار": "r",
        "تي": "t",

        # Suffixes
        "برو": "pro",
        "بلس": "plus",
        "الترا": "ultra",
        "اوبن": "open",
        "ماكلارين": "mclaren",
    },

    "pre": [
        # Normalize brand spelling
        (r"\bone\s+plus\b", "oneplus"),
        (r"\boneplus(?=[a-z0-9])", "oneplus "),

        # Normalize series names
        (r"\bace\s*(?=\d)", "ace "),
        (r"\bnord\s+ce\s*(?=\d)", "nord ce "),
        (r"\bnord\s*(?=\d)", "nord "),

        # Separate suffixes attached to digits
        (r"(?<=\d)(pro|plus|ultra)\b", r" \1"),

        # Normalize numbered suffixes
        (r"\b(\d{1,2})\s*t\s*pro\s*mclaren\b",
         r"\1t pro mclaren"),

        (r"\b(\d{1,2})\s*t\s*pro\b",
         r"\1t pro"),

        (r"\b(\d{1,2})\s*r\b",
         r"\1r"),
    ],

    "rules": [
        # -------------------------
        # OnePlus Ace
        # -------------------------
        (r"\bace\s*(\d{1,2})\s*t\b",
         "ace {1}t"),

        (r"\bace\s*(\d{1,2})\s*(pro|plus|ultra)\b",
         "ace {1} {2}"),

        (r"\bace\s*(\d{1,2})\b",
         "ace {1}"),

        # -------------------------
        # OnePlus Nord CE
        # -------------------------
        (r"\bnord\s*ce\s*(\d{1,2})\s*(lite)?\b",
         "nord ce{1} {2}"),

        # -------------------------
        # OnePlus Nord
        # -------------------------
        (r"\bnord\s*(\d{1,2})\s*(pro|plus)?\b",
         "nord {1} {2}"),

        # -------------------------
        # OnePlus Open
        # -------------------------
        (r"\bopen(?:\s+fold)?\b",
         "open"),

        # -------------------------
        # OnePlus numbered models
        # -------------------------

        # 7T Pro McLaren Edition
        (r"\b(\d{1,2})\s*t\s*pro\s*mclaren\b",
         "{1}t pro mclaren"),

        # 13s, 13t, 12r
        (r"\b(\d{1,2})\s*(s|t|r)\b",
         "{1}{2}"),

        # Numbered Pro models
        (r"\b(\d{1,2})\s*pro\b",
         "{1} pro"),

        # Numbered Plus / Ultra models
        (r"\b(\d{1,2})\s*(plus|ultra)\b",
         "{1} {2}"),

        # Numbered models without suffix
        (r"\b(\d{1,2})\b",
         "{1}"),
    ],
}

In [ ]:
evaluate_brand("oneplus")

Brand: oneplus
Total ads: 17
Models extracted: 17
Unknown models: 0
Extraction coverage: 100.00%


# zte

In [ ]:
explore_brand("zte")

Total zte ads: 17


,title,title_punct_normalized
758,موبايل Nubia A76 5G جديد للبيع,موبايل nubia a76 5g جديد للبيع
463,زد تى إي نيو 2,زد تي اي نيو 2
4646,nubia neo 5g,nubia neo 5g
1307,Nubia v60,nubia v60
911,زد تى إي نوبيا زد 30 برو,زد تي اي نوبيا زد 30 برو
1528,موبايل نوبيا A36,موبايل نوبيا a36
2077,تليفون نوبيا كسر زيرو بيع بسعر حرق,تليفون نوبيا كسر زيرو بيع بسعر حرق
180,زد تى إي نوبيا زد 30 برو,زد تي اي نوبيا زد 30 برو
3458,نوبيا A36,نوبيا a36
4647,ZTE Other Model,zte other model


In [ ]:
BRAND_CONFIG["zte"] = {
    "letters": {},

    "aliases": {
        # Brand
        "زد تي اي": "zte",
        "زد تى اي": "zte",
        "زد تي إي": "zte",
        "زد تى إي": "zte",
        "زدتي اي": "zte",
        "زدتي": "zte",

        # Nubia
        "نوبيا": "nubia",
        "نوبيا زد": "nubia z",

        # RedMagic
        "ريد ماجيك": "red magic",
        "ريدماجيك": "red magic",

        # Neo
        "نيو": "neo",

        # Other
        "برو": "pro",
        "اير": "air",
        "إير": "air",
        "الترا": "ultra",
        "بلس": "plus",
    },

    "pre": [
        # Brand normalization
        (r"\bzte(?=[a-z0-9])", "zte "),

        # Normalize Nubia and RedMagic
        (r"\bred\s*magic\s*(?=\d)", "red magic "),
        (r"\bnubia\s*z\s*(?=\d)", "nubia z "),
        (r"\bnubia\s*a\s*(?=\d)", "nubia a "),
        (r"\bnubia\s*v\s*(?=\d)", "nubia v "),
        (r"\bnubia\s*neo\s*(?=\d)", "nubia neo "),

        # Blade series
        (r"\bblade\s*a\s*(?=\d)", "blade a "),

        # Normalize Neo series
        (r"\bneo\s*(?=\d)", "neo "),

        # Fix attached model names
        (r"\bnubia(?=[a-z]\d)", "nubia "),
    ],

    "rules": [
        # -------------------------
        # RedMagic
        # -------------------------
        (r"\bred\s*magic\s*(\d{1,2})\s*(pro|max|ultra|air)?\b",
         "red magic {1} {2}"),

        # -------------------------
        # Nubia Neo
        # -------------------------
        (r"\bnubia\s*neo\s*(\d{1,2})\s*(5g|pro|plus)?\b",
         "nubia neo {1} {2}"),

        # -------------------------
        # Nubia Z series
        # -------------------------
        (r"\bnubia\s*z\s*(\d{1,2})\s*(pro|ultra|plus)?\b",
         "nubia z{1} {2}"),

        # -------------------------
        # Nubia A series
        # -------------------------
        (r"\bnubia\s*a\s*(\d{1,3})\s*(5g)?\b",
         "nubia a{1} {2}"),

        # -------------------------
        # Nubia V series
        # -------------------------
        (r"\bnubia\s*v\s*(\d{1,3})\s*(5g)?\b",
         "nubia v{1} {2}"),

        # -------------------------
        # ZTE Blade
        # -------------------------
        (r"\bblade\s*a\s*(\d{1,3})\s*(5g)?\b",
         "blade a{1} {2}"),

        # -------------------------
        # ZTE Neo
        # -------------------------
        (r"\bneo\s*(\d{1,2})\b",
         "nubia neo {1}"),

        # -------------------------
        # Unknown / Other Model
        # -------------------------
        (r"\bother\s+model\b",
         "unknown"),
    ],
}

In [ ]:
evaluate_brand("zte")

Brand: zte
Total ads: 17
Models extracted: 15
Unknown models: 2
Extraction coverage: 88.24%


In [ ]:
stage1_df.loc[
    stage1_df["zte_model"].eq("unknown"),
    ["title", "title_normalized", "zte_model"]
].head(69)

,title,title_normalized,zte_model
2077,تليفون نوبيا كسر زيرو بيع بسعر حرق,تليفون نوبيا كسر زيرو بيع بسعر حرق,unknown
4646,nubia neo 5g,nubia neo 5g,unknown


# tecno

In [ ]:
explore_brand("tecno")

Total tecno ads: 15


,title,title_punct_normalized
964,تكنو سبارك 40 برو جديد,تكنو سبارك 40 برو جديد
2540,تكنو بوفا 6 برو للتبديل بايفون 12 برو,تكنو بوفا 6 برو للتبديل بايفون 12 برو
4613,تكنو كامون 30 للبيع داخل محافظة المنيا,تكنو كامون 30 للبيع داخل محافظة المنيا
2305,Tecno Spark 20 Pro +,tecno spark 20 pro +
2960,Tecno Pop 7,tecno pop 7
321,تكنو سبارك جو ٣ جديد ما اشتغلش لسه جديد شراء ا...,تكنو سبارك جو 3 جديد ما اشتغلش لسه جديد شراء ا...
3595,تكنو سبارك سي إم كسر زيرو مفيش خربوش256/16,تكنو سبارك سي ام كسر زيرو مفيش خربوش256/16
1532,Tecno Pova 7,tecno pova 7
4330,تليفون جديد تكنو سبارك 30C مساحه 256 رام معاه ...,تليفون جديد تكنو سبارك 30c مساحه 256 رام معاه ...
4851,Tecno Spark 20 Pro + السعر نهائي,tecno spark 20 pro + السعر نهائي


In [ ]:
BRAND_CONFIG["tecno"] = {
    "letters": {},

    "aliases": {
        "تكنو": "tecno",

        "سبارك": "spark",
        "سبارك جو": "spark go",
        "سبارك سي ام": "spark cm",
        "سبارك سي إم": "spark cm",

        "كامون": "camon",
        "كامون سي ام": "camon cm",
        "كامون سي إم": "camon cm",

        "بوفا": "pova",
        "بوب": "pop",

        "برو": "pro",
        "بلس": "plus",
        "جو": "go",
    },

    "pre": [
        (r"\btecno(?=[a-z0-9])", "tecno "),

        (r"\bspark\s*go\s*(?=\d)", "spark go "),
        (r"\bspark\s*cm\b", "spark cm"),
        (r"\bspark\s*(?=\d)", "spark "),
        (r"\bcamon\s*cm\b", "camon cm"),
        (r"\bcamon\s*(?=\d)", "camon "),
        (r"\bpova\s*(?=\d)", "pova "),
        (r"\bpop\s*(?=\d)", "pop "),

        (r"(?<=\d)(pro|plus)\b", r" \1"),
    ],

    "rules": [
        # Tecno Spark Go 3
        (
            r"\bspark\s*go\s*(\d{1,2})\b",
            "spark go {1}"
        ),

        # Tecno Spark 20 Pro+
        (
            r"\bspark\s*(\d{1,3})\s*pro\s*plus\b",
            "spark {1} pro plus"
        ),

        # Tecno Spark 20 Pro
        (
            r"\bspark\s*(\d{1,3})\s*pro\b",
            "spark {1} pro"
        ),

        # Tecno Spark CM
        (
            r"\bspark\s*cm\b",
            "spark cm"
        ),

        # Tecno Spark 30C
        (
            r"\bspark\s*(\d{1,3})\s*c\b",
            "spark {1}c"
        ),

        # Tecno Spark 30 / Spark 40
        (
            r"\bspark\s*(\d{1,3})\b",
            "spark {1}"
        ),

        # Tecno Camon CM
        (
            r"\bcamon\s*cm\b",
            "camon cm"
        ),

        # Tecno Camon 40 Pro 5G
        (
            r"\bcamon\s*(\d{1,3})\s*pro\s*5g\b",
            "camon {1} pro 5g"
        ),

        # Tecno Camon 40 Pro
        (
            r"\bcamon\s*(\d{1,3})\s*pro\b",
            "camon {1} pro"
        ),

        # Tecno Camon 40
        (
            r"\bcamon\s*(\d{1,3})\b",
            "camon {1}"
        ),

        # Tecno Pova 6 Pro / Pova 7
        (
            r"\bpova\s*(\d{1,3})\s*(pro|plus|ultra)?\b",
            "pova {1} {2}"
        ),

        # Tecno Pop 7
        (
            r"\bpop\s*(\d{1,3})\b",
            "pop {1}"
        ),
    ],
}

In [ ]:
evaluate_brand("tecno")

Brand: tecno
Total ads: 15
Models extracted: 13
Unknown models: 2
Extraction coverage: 86.67%


# itel

In [ ]:
explore_brand("itel")

Total itel ads: 12


,title,title_punct_normalized
1448,ايتل A90,ايتل a90
3237,ايتل A100c,ايتل a100c
3603,ايتل موديل اخر,ايتل موديل اخر
3232,itel A90,itel a90
4684,ايتل a90,ايتل a90
3776,ايتل ايه 80,ايتل ايه 80
4368,ايتل اس 24,ايتل اس 24
1743,ايتل ايه 50 سى,ايتل ايه 50 سي
1144,ايتل موديل اخر,ايتل موديل اخر
1079,Itel A90 (ايتيل ايه ٩٠),itel a90 (ايتيل ايه 90)


In [ ]:

BRAND_CONFIG["itel"] = {
    "letters": {},
    "aliases": {
        "ايتل": "itel",
        "ايتيل": "itel",

        "اس": "s",
        "إس": "s",

        "ايه": "a",
        "إيه": "a",

        "ار": "r",
        "آر": "r",

        "اس 4": "s4",
        "اس 24": "s24",
        "اس 25": "s25",

        "الترا": "ultra",
        "سي": "c",
        "سى": "c",
    },

    "pre": [
        (r"\bitel(?=[a-z0-9])", "itel "),
        (r"\b([as])\s*(\d{1,3})\s*c\b", r"\1 \2c"),
        (r"\ba\s*(?=\d)", "a "),
        (r"\bs\s*(?=\d)", "s "),
        (r"\brs\s*(?=\d)", "rs "),
        (r"\br\s+s\s*(?=\d)", "rs "),
        (r"\b(\d{1,3})\s*c\b", r"\1c"),
    ],

    "rules": [
        # itel S25 Ultra
        (
            r"\bs\s*(\d{1,3})\s*ultra\b",
            "s{1} ultra"
        ),

        # itel S24 / S25
        (
            r"\bs\s*(\d{1,3})\b",
            "s{1}"
        ),

        # itel A50C
        (
            r"\ba\s*(\d{1,3})\s*c\b",
            "a{1}c"
        ),

        # itel A90 / A80 / A100C
        (
            r"\ba\s*(\d{1,3})\s*c\b",
            "a{1}c"
        ),
        (
            r"\ba\s*(\d{1,3})\b",
            "a{1}"
        ),

        # itel RS4
        (
            r"\brs\s*(\d{1,3})\b",
            "rs {1}"
        ),
    ],
}


In [ ]:
evaluate_brand("itel")

Brand: itel
Total ads: 12
Models extracted: 10
Unknown models: 2
Extraction coverage: 83.33%


In [ ]:
explore_brand("google")

Total google ads: 13


,title,title_punct_normalized
2175,جوجل بكسل 8 ايه,جوجل بكسل 8 ايه
2708,للبيع موبايل جوجل بكسل 6 برو,للبيع موبايل جوجل بكسل 6 برو
4487,Pixel 11 Pro Fold,pixel 11 pro fold
2606,جوجل بكسل 6 ايه,جوجل بكسل 6 ايه
2815,Google Pixel 7,google pixel 7
1546,Google Pixel 6 Pro,google pixel 6 pro
5089,Google pixel 9a مدفوع الضريبة,google pixel 9a مدفوع الضريبة
2583,جوجل بكسل 6 برو,جوجل بكسل 6 برو
3624,تليفون google pixel 7 pro بالعلبه وكامل المشتملات,تليفون google pixel 7 pro بالعلبه وكامل المشتملات
567,Google Pixel 6 pro 12-128,google pixel 6 pro 12 128


In [ ]:

BRAND_CONFIG["google"] = {
    "letters": {},
    "aliases": {
        "جوجل": "google",
        "بكسل": "pixel",
        "برو": "pro",
        "فولد": "fold",
        "ايه": "a",
        "إيه": "a",
    },

    "pre": [
        (r"\bgoogle(?=[a-z0-9])", "google "),
        (r"\bpixel\s*(?=\d)", "pixel "),
        (r"\b(\d{1,2})\s*pro\s*fold\b", r"\1 pro fold"),
        (r"\b(\d{1,2})\s*pro\b", r"\1 pro"),
        (r"\b(\d{1,2})\s*5g\b", r"\1 5g"),
    ],

    "rules": [
        # Pixel 11 Pro Fold
        (
            r"\bpixel\s*(\d{1,2})\s*pro\s*fold\b",
            "{1} pro fold"
        ),

        # Pixel 6 Pro / Pixel 9 Pro
        (
            r"\bpixel\s*(\d{1,2})\s*pro\b",
            "{1} pro"
        ),

        # Pixel 8a / Pixel 9a
        (
            r"\bpixel\s*(\d{1,2})\s*a\b",
            "{1}a"
        ),

        # Pixel 10 5G
        (
            r"\bpixel\s*(\d{1,2})\s*5g\b",
            "{1} 5g"
        ),

        # Pixel 6 / Pixel 7 / Pixel 10 / Pixel 11
        (
            r"\bpixel\s*(\d{1,2})\b",
            "{1}"
        ),
    ],
}


In [ ]:
evaluate_brand("google")

Brand: google
Total ads: 13
Models extracted: 13
Unknown models: 0
Extraction coverage: 100.00%


# sony
11
nokia
9
lenovo
8
lg

In [ ]:
explore_brand("sony")

Total sony ads: 7


,title,title_punct_normalized
847,Sony Other Model,sony other model
3583,سوني اريكسون K610i,سوني اريكسون k610i
235,شاشه touchscreen panel sony xperia Z Ultra للبيع,شاشه touchscreen panel sony xperia z ultra للبيع
918,سوني زد 2,سوني زد 2
1417,سوني اكسبريا 4مارك 5,سوني اكسبريا 4مارك 5
3033,Sony xperia 1 mark 7 سوني إكسبريا ١ مارك ٧,sony xperia 1 mark 7 سوني اكسبريا 1 مارك 7
715,Sony Xperia 1 mark 2. كسر زيرو بالعلبة كاملة,sony xperia 1 mark 2. كسر زيرو بالعلبة كاملة


In [ ]:

BRAND_CONFIG["sony"] = {
    "letters": {},
    "aliases": {
        "سوني": "sony",
        "اكسبريا": "xperia",
        "إكسبريا": "xperia",
        "اريكسون": "ericsson",
        "إريكسون": "ericsson",
        "مارك": "mark",
        "برو": "pro",
        "الترا": "ultra",
        "زد": "z",
    },

    "pre": [
        (r"\bsony(?=[a-z0-9])", "sony "),
        (r"\bxperia(?=[a-z0-9])", "xperia "),
        (r"\b(\d)\s*mark\s*(\d)\b", r"\1 mark \2"),
        (r"\b(\d{1,2})\s*(iv|v|vi|iii|ii)\b", r"\1 \2"),
        (r"\bpro\s*[- ]?\s*i\b", "pro i"),
        (r"\bz\s*ultra\b", "z ultra"),
    ],

    "rules": [
        # Sony Xperia 1 Pro-I
        (
            r"\bxperia\s+pro\s+i\b",
            "xperia pro-i"
        ),

        # Sony Xperia 1 Mark 2 / Mark 7
        (
            r"\bxperia\s+(\d)\s*mark\s*(\d)\b",
            "xperia {1} mark {2}"
        ),

        # Sony Xperia 1 V / 5 IV
        (
            r"\bxperia\s+(\d{1,2})\s*(iv|v|vi|iii|ii)\b",
            "xperia {1} {2}"
        ),

        # Sony Xperia Z Ultra
        (
            r"\bxperia\s+z\s+ultra\b",
            "xperia z ultra"
        ),

        # Sony Xperia 1 / 5
        (
            r"\bxperia\s+(\d{1,2})\b",
            "xperia {1}"
        ),

        # Sony Ericsson K610i
        (
            r"\b([a-z]{1,3}\d{2,4}i?)\b",
            "{1}"
        ),
    ],
}


In [ ]:
evaluate_brand("sony")

Brand: sony
Total ads: 7
Models extracted: 5
Unknown models: 2
Extraction coverage: 71.43%


# nokia

In [ ]:
explore_brand("nokia")

Total nokia ads: 8


,title,title_punct_normalized
1249,Nokia G300 5G وارد أمريكا بلمعته كالجديد شاشه...,nokia g300 5g وارد امريكا بلمعته كالجديد شاشه ...
1823,Nokia Other Model,nokia other model
2773,Nokia N9,nokia n9
3886,نوكيا موديل اخر,نوكيا موديل اخر
116,Nokia C21 Plus,nokia c21 plus
2118,نوكيا اكس 2,نوكيا اكس 2
2460,NOKIA X350 for sale,nokia x350 for sale
163,نوكيا 7.1 للبيع,نوكيا 7.1 للبيع


In [ ]:

BRAND_CONFIG["nokia"] = {
    "letters": {},
    "aliases": {
        "نوكيا": "nokia",
        "اكس": "x",
        "إكس": "x",
        "بلس": "plus",
        "برو": "pro",
        "الترا": "ultra",
    },

    "pre": [
        (r"\bnokia(?=[a-z0-9])", "nokia "),

        (r"\b([gc])\s*(?=\d)", r"\1 "),
        (r"\bx\s*(?=\d)", "x "),
        (r"\bn\s*(?=\d)", "n "),

        (r"\b(\d+)\s*\.\s*(\d+)\b", r"\1.\2"),

        (r"(?<=\d)(plus|pro|ultra)\b", r" \1"),
    ],

    "rules": [
        # Nokia G300 5G
        (
            r"\bg\s*(\d{1,3})\s*5g\b",
            "g{1} 5g"
        ),
        (
            r"\bg\s*(\d{1,3})\b",
            "g{1}"
        ),

        # Nokia C21 Plus / C-series
        (
            r"\bc\s*(\d{1,3})\s*plus\b",
            "c{1} plus"
        ),
        (
            r"\bc\s*(\d{1,3})\s*pro\b",
            "c{1} pro"
        ),
        (
            r"\bc\s*(\d{1,3})\b",
            "c{1}"
        ),

        # Nokia X350 / X2
        (
            r"\bx\s*(\d{1,4})\b",
            "x{1}"
        ),

        # Nokia N9
        (
            r"\bn\s*(\d{1,3})\b",
            "n{1}"
        ),

        # Nokia 7.1
        (
            r"\b(\d{1,2})\.(\d)\b",
            "{1}.{2}"
        ),
    ],
}




In [ ]:
evaluate_brand("nokia")

Brand: nokia
Total ads: 8
Models extracted: 6
Unknown models: 2
Extraction coverage: 75.00%


# lenovo

In [ ]:
explore_brand("lenovo")

Total lenovo ads: 6


,title,title_punct_normalized
3412,لينوفو نوت6 بالعلبه 32ج 3 رام ماتفتحش خدش فى ا...,لينوفو نوت6 بالعلبه 32ج 3 رام ماتفتحش خدش في ا...
4634,Lenovo Other Model,lenovo other model
1580,Lenovo Legion Y70 2026 mobile phone,lenovo legion y70 2026 mobile phone
3680,lenovo k15 plus(لسه موجود),lenovo k15 plus(لسه موجود)
4505,موبايل لينوفو Lenovo K8 NOTE,موبايل لينوفو lenovo k8 note
1649,Lenovo A8,lenovo a8


In [ ]:

BRAND_CONFIG["lenovo"] = {
    "letters": {},
    "aliases": {
        "لينوفو": "lenovo",
        "كيه": "k",
        "نوت": "note",
        "نوتة": "note",
        "ليجن": "legion",
        "ليجيون": "legion",
        "بلس": "plus",
        "برو": "pro",
    },

    "pre": [
        (r"\blenovo(?=[a-z0-9])", "lenovo "),
        (r"\blegion\s*y\s*(?=\d)", "legion y "),
        (r"\bk\s*(?=\d)", "k "),
        (r"\ba\s*(?=\d)", "a "),
        (r"\b(note)\s*(?=\d)", "note "),
        (r"\b(\d{1,3})\s*note\b", r"\1 note"),
    ],

    "rules": [
        # Lenovo Legion Y70
        (
            r"\blegion\s*y\s*(\d{1,3})\b",
            "legion y{1}"
        ),

        # Lenovo K14 Plus
        (
            r"\bk\s*(\d{1,3})\s*plus\b",
            "k{1} plus"
        ),

        # Lenovo K12 Note / K8 Note
        (
            r"\bk\s*(\d{1,3})\s*note\b",
            "k{1} note"
        ),

        # Lenovo K15
        (
            r"\bk\s*(\d{1,3})\b",
            "k{1}"
        ),

        # Lenovo A8
        (
            r"\ba\s*(\d{1,3})\b",
            "a{1}"
        ),

        # Lenovo Note 6
        (
            r"\bnote\s*(\d{1,3})\b",
            "note {1}"
        ),
    ],
}


In [ ]:
evaluate_brand("lenovo")

Brand: lenovo
Total ads: 6
Models extracted: 5
Unknown models: 1
Extraction coverage: 83.33%


In [ ]:
explore_brand("lg")

Total lg ads: 3


,title,title_punct_normalized
591,lg g6 خطين محتاج تاتش,lg g6 خطين محتاج تاتش
4994,lg g6 جديد خطين,lg g6 جديد خطين
4320,موبايل Lg,موبايل lg


In [ ]:

BRAND_CONFIG["lg"] = {
    "letters": {},
    "aliases": {
        "ال جي": "lg",
        "إل جي": "lg",
        "ال جى": "lg",
        "إل جى": "lg",

        "فيلفيت": "velvet",
        "وينج": "wing",
        "وينغ": "wing",
        "ستايلو": "stylo",
        "ثينك كيو": "thinq",
        "ثينكيو": "thinq",
    },

    "pre": [
        (r"\blg(?=[a-z0-9])", "lg "),

        (r"\b(g|v|k|q|w|x)\s*(?=\d)", r"\1 "),
        (r"\bstylo\s*(?=\d)", "stylo "),
        (r"\bvelvet(?=\d)", "velvet "),
        (r"\bwing(?=\d)", "wing "),

        (r"\bthin\s*q\b", "thinq"),
        (r"\bthinq\b", "thinq"),

        (r"\b(v\d{1,2})\s*thinq\b", r"\1 thinq"),
    ],

    "rules": [
        # LG Velvet
        (r"\bvelvet\s*(\d{1,2})?\b", "velvet {1}"),

        # LG Wing
        (r"\bwing\s*(\d{1,2})?\b", "wing {1}"),

        # LG Stylo 6 / Stylo 7
        (r"\bstylo\s*(\d{1,2})\b", "stylo {1}"),

        # LG V60 ThinQ / V50 ThinQ
        (r"\bv\s*(\d{1,2})\s*thinq\b", "v{1} thinq"),

        # LG G-series
        (r"\bg\s*(\d{1,2})\b", "g{1}"),

        # LG V-series
        (r"\bv\s*(\d{1,2})\b", "v{1}"),

        # LG K-series
        (r"\bk\s*(\d{1,3})\b", "k{1}"),

        # LG Q-series
        (r"\bq\s*(\d{1,3})\b", "q{1}"),

        # LG W-series
        (r"\bw\s*(\d{1,3})\b", "w{1}"),

        # LG X-series
        (r"\bx\s*(\d{1,3})\b", "x{1}"),
    ],
}


In [ ]:
evaluate_brand("lg")

Brand: lg
Total ads: 3
Models extracted: 2
Unknown models: 1
Extraction coverage: 66.67%


In [ ]:
print("Columns names:\n", stage1_df.columns.to_list())

Columns names:
 ['title', 'price', 'location', 'ad_url', 'ad_id', 'run_id', 'brand', 'storage_gb', 'ram_gb', 'storage_src', 'condition', 'condition_src', 'ram_src', 'has_5g', 'has_warranty', 'has_box', 'mentions_exchange', 'is_wanted_ad', 'is_accessory_hint', 'is_tablet_hint', 'is_parts_hint', 'accessory_first', 'bundle_hint', 'is_feature_phone_hint', 'price_placeholder', 'loc_area', 'loc_city', 'price_missing', 'price_suspicious', 'price_very_high', 'title_normalized', 'title_punct_normalized', 'title_group', 'iphone_model', 'samsung_model', 'is_phone_listing', 'xiaomi_model', 'oppo_model', 'honor_model', 'realme_model', 'infinix_model', 'vivo_model', 'huawei_model', 'motorola_model', 'oneplus_model', 'zte_model', 'tecno_model', 'itel_model', 'google_model', 'sony_model', 'nokia_model', 'lenovo_model', 'lg_model']


In [ ]:
def apply_stage1_extraction(data):
    data = data.copy()
    # 1. Apple
    apple_mask = data["brand"] == "apple"
    data["iphone_model"] = "not_apple"

    data.loc[apple_mask, "iphone_model"] = (
        data.loc[apple_mask, "title_punct_normalized"]
        .apply(extract_iphone_model)
    )
    # 2. باقي الماركات: كل ماركة تستخدم قواعدها الخاصة
    for brand in BRAND_CONFIG:

        col = f"{brand}_model"
        brand_mask = data["brand"] == brand

        data[col] = pd.Series(index=data.index, dtype="object")

        # Samsung كان بيستخدم title_punct_normalized في كودك
        if brand == "samsung":
            source_col = "title_punct_normalized"
        else:
            source_col = "title"

        data.loc[brand_mask, col] = (
            data.loc[brand_mask, source_col]
            .apply(lambda title: extract_brand_model(title, brand))
        )

    # 3. تحديد إعلانات قطع غيار Samsung
    parts_only_pattern = (
        r"^\s*(?:شاشه|شاشة|بطاريه|بطارية|بورده|بوردة|"
        r"جراب|اسكرينه|اسكرينة|شاحن|سماعه|سماعة)\b"
        r"|محتاج شاشه|محتاج شاشة|مطلوب شاشه|مطلوب شاشة"
        r"|screen\s+(?:samsung|for\s+samsung)"
    )

    data["is_phone_listing"] = True

    parts_mask = (
        (data["brand"] == "samsung")
        & data["title"].str.contains(
            parts_only_pattern,
            case=False,
            na=False,
            regex=True
        )
    )

    data.loc[parts_mask, "is_phone_listing"] = False

    return data

In [ ]:
stage1_val_df = apply_stage1_extraction(stage1_val_df)

stage1_test_df = apply_stage1_extraction(stage1_test_df)

print("Validation shape:", stage1_val_df.shape)
print("Test shape:", stage1_test_df.shape)

Validation shape: (1252, 53)
Test shape: (1796, 53)


In [ ]:
new_columns = [
    col for col in stage1_df.columns
    if col not in train_df.columns
]
print("New columns in Train:", new_columns)

print("\nMissing from Validation:")
print(set(new_columns) - set(stage1_val_df.columns))

print("\nMissing from Test:")
print(set(new_columns) - set(stage1_test_df.columns))

New columns in Train: ['iphone_model', 'samsung_model', 'is_phone_listing', 'xiaomi_model', 'oppo_model', 'honor_model', 'realme_model', 'infinix_model', 'vivo_model', 'huawei_model', 'motorola_model', 'oneplus_model', 'zte_model', 'tecno_model', 'itel_model', 'google_model', 'sony_model', 'nokia_model', 'lenovo_model', 'lg_model']

Missing from Validation:
set()

Missing from Test:
set()
